# Análise individual: plano dos Estados Unidos (*America’s AI Action Plan*)

**Projeto:** *As Relações Brasil-China e a Política de Governança Brasileira sobre Inteligência Artificial* (FAPESP, Processo nº 2026/04597-1)
**Etapa:** 02 (análise individual), conforme a skill 02 ([`skill_análise_individual.md`](skill_análise_individual.md))
**Insumo:** `EUA_2025_extracao.json`, produzido pela skill 01 ([`skill_extração.md`](skill_extração.md))

---

> ⚠️ **Visualizações prévias.** Todas as visualizações geradas neste notebook (gráficos e tabelas) são **prévias** e compõem **apenas uma base para o início da construção de novos gráficos**. As novas visualizações serão construídas de acordo com os comandos e as exigências do pesquisador, que são obedecidos em última instância. Nenhuma visualização, tabela ou relatório vai para a pasta `resultados/` sem comando expresso do pesquisador.

## 1. Do que se trata

Este notebook analisa, de forma **individual**, o texto do plano de inteligência artificial dos Estados Unidos, *America’s AI Action Plan* (subtítulo: *Winning the Race*). Ele faz parte do fluxo de análise da pasta *Planos Nacionais de IA*: a skill 01 extrai o texto essencial de cada plano; esta etapa (skill 02) analisa cada plano isoladamente; a skill 03 compara os planos entre si.

Por comando do pesquisador, este notebook segue o **mesmo molde** das análises individuais da pasta *Declarações IA* (`Análise_OCDE.ipynb` e `Análise_BRICS.ipynb`): os mesmos itens, na mesma ordem, até as expressões recorrentes (bigramas), com as adaptações exigidas pela skill desta pasta. Daqueles notebooks aproveitou-se **apenas o molde** (estrutura, código e textos de método); nenhum dado, resultado ou interpretação deles é usado aqui.

## 2. Finalidade

A finalidade deste notebook é **gerar visualizações e interpretações sobre o plano analisado**, por meio de gráficos, tabelas e medidas textuais. As análises permitem conhecer:

- a **extensão** e a estrutura do texto;
- os **termos mais comuns** e como se distribuem ao longo do documento;
- as **expressões recorrentes**.

## 3. Insumo e recorte do conteúdo

São utilizados **apenas os resultados da skill 01**, ou seja, o arquivo JSON do plano gerado por aquela etapa. O relatório de extração (`relatorios_extracao/`) é apenas informativo e **não** é consultado. Do JSON, interessam **somente**:

- o **título** e o **subtítulo** do plano (`document.title` e `document.subtitle`), que identificam o documento e, por padrão, **entram na contagem** como unidade 0 (`INCLUIR_TITULO_NA_CONTAGEM = True`), porque fazem parte do texto essencial do plano;
- o **texto integral e literal de cada unidade** (`sections[].text`): títulos internos, parágrafos e itens de lista. É o conteúdo analisado;
- o **tipo** e a **categoria estrutural** de cada unidade (`sections[].type` e `sections[].category`), usados apenas como **filtros**. Por padrão, entram os títulos internos (`INCLUIR_TITULOS_INTERNOS = True`) e todas as categorias (`CATEGORIAS_EXCLUIDAS = []`);
- o bloco `extraction`, usado apenas para **verificar** a integridade do arquivo.

O identificador de cada unidade (`id`) é usado apenas como índice de posição no documento. **Todo conteúdo periférico fica fora da análise**: numeração e marcadores (`number`), nível dos títulos (`level`), página (`page`), caminho estrutural (`path`) e os demais metadados (emissor, data, período etc.). A função `carregar_extracao` descarta esse conteúdo no momento da leitura. Todo filtro aplicado é informado no perfil do documento, com as unidades e os tokens que retirou.

## 4. Método (visão geral)

| Etapa | Procedimento |
|---|---|
| Verificação | Integridade do JSON, origem na skill 01 para o corpus *Planos Nacionais de IA* e exame de sinais de inflação (unidades ou frases repetidas, resíduos de extração) |
| Pré-processamento | spaCy (`en_core_web_sm`): tokenização, reunião de compostos com hífen, lematização com correções explícitas, classes gramaticais, *stopwords* |
| Frequências | Frequência absoluta, por 1.000 tokens, alcance (unidades em que o termo aparece) e dispersão (DP normalizado de Gries) |
| Expressões | Bigramas de tokens de conteúdo adjacentes no texto original |

O detalhamento de cada procedimento está na configuração, nas funções comentadas e no texto que acompanha cada visualização.

## 5. Transparência das escolhas

**Todo método utilizado e todas as escolhas tomadas pelo pesquisador estão presentes e verificáveis** em dois lugares:

1. **no código**: os parâmetros ficam reunidos na célula de configuração, e cada função traz comentários sobre o que faz;
2. **na descrição que segue cada visualização**, composta por: *Método e escolhas*, *Como ler*, *Interpretação* e *Fragilidades e ajustes possíveis*.

As tabelas completas (e não apenas os recortes exibidos nos gráficos), as figuras e os demais arquivos **só são exportados para a pasta `resultados/` por comando expresso do pesquisador** (skill 02, seção 4.4). Sem esse comando, o notebook apenas exibe os resultados.

## 6. Princípios

Todo o conteúdo deste notebook segue os princípios da **skill 02** (análise individual dos planos). Por isso, **não incorpora análise conjunta**: nenhum resultado aqui é comparado com outro plano. A comparação entre planos é feita em notebook próprio, segundo a skill 03.

> As interpretações são redigidas **somente após a execução** do notebook, com base nos valores efetivamente obtidos, separando observação de inferência. Se o notebook for executado de novo com outros parâmetros ou com outro JSON, as interpretações precisam ser revistas.

## 7. Configuração

Todos os parâmetros editáveis estão nesta célula. Qualquer alteração deve ser registrada. A gravação em `resultados/` está desativada (`SALVAR_SAIDAS = False`) e só é ativada por comando do pesquisador.

In [ ]:
import hashlib
import json
import math
import platform
import re
import unicodedata
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spacy
from IPython.display import display
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter, MaxNLocator

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 140)

# =====================================================================
# CONFIGURAÇÃO — parâmetros editáveis pelo pesquisador
# =====================================================================
SIGLA = "EUA_2025"
ARQUIVO_JSON = "EUA_2025_extracao.json"        # entregável da skill 01 (somente leitura)

# Saídas (figuras, tabelas). Resultados anteriores nunca são sobrescritos:
# para uma nova execução com mudanças, altere VERSAO_SAIDA (v2, v3, ...).
NOME_SAIDA = SIGLA
VERSAO_SAIDA = "v1"
# Gravação em resultados/ DESATIVADA: visualizações, tabelas e relatórios só vão para
# resultados/ por comando expresso do pesquisador (skill 02, seção 4.4).
SALVAR_SAIDAS = False
SOBRESCREVER = False

# Recorte do conteúdo (skill 02, seção 3.1)
INCLUIR_TITULO_NA_CONTAGEM = True    # título e subtítulo entram nas contagens como unidade 0
INCLUIR_TITULOS_INTERNOS = True      # unidades do tipo "heading" (pilares e seções)
CATEGORIAS_EXCLUIDAS = []            # ex.: ["Foreword", "Executive summary", "Box", "Annex"]
UNIDADES_EXCLUIDAS = {}              # {id: "motivo"}: tratamento de inflação (skill 02, seção 3.5)

# Pré-processamento (skill 02, seção 3.3)
IDIOMA_ESPERADO = "en"
MODELO_SPACY = "en_core_web_sm"
CLASSES_DE_CONTEUDO = {"NOUN", "PROPN", "VERB", "ADJ", "ADV"}
TAMANHO_MINIMO_LEMA = 2
STOPWORDS_ADICIONAIS = []    # lemas a remover além da lista do spaCy (decisão do pesquisador)
STOPWORDS_MANTIDAS = []      # lemas da lista do spaCy que devem ser mantidos (decisão do pesquisador)
CORRECOES_DE_LEMA = {
    # Plurais latinos que o spaCy reduz a um singular que não é a palavra usada no texto.
    "datum": "data",         # "data" -> "datum"; mantém-se "data", forma usada no texto
    "medium": "media",       # "media" (meios de comunicação) -> "medium"; no texto, "medium" só
                             # aparece no composto "medium-sized", que é reunido em um token
    # Títulos internos em maiúsculas iniciais (Title Case): o spaCy etiqueta essas palavras
    # como nome próprio (PROPN) e não as lematiza; no corpo do texto, as mesmas palavras são
    # lematizadas normalmente. A correção alinha os títulos ao corpo e evita que um mesmo
    # termo seja dividido em dois lemas (ex.: "risks" nos títulos e "risk" no corpo).
    "allies": "ally",
    "applications": "application",
    "breakthroughs": "breakthrough",
    "centers": "center",     # também une 2 ocorrências do corpo em nomes próprios (AI Centers of
                             # Excellence; Research and Development Centers), plural de "center"
    "evaluations": "evaluation",
    "existing": "exist",     # no corpo, o spaCy lematiza "existing" como "exist"
    "facilities": "facility",
    "measures": "measure",
    "protects": "protect",
    "risks": "risk",
    "technologies": "technology",
    "workers": "worker",
}

# Recortes de visualização (as tabelas exportadas são sempre completas)
TOP_N_LEMAS = 25
TOP_N_DISPERSAO = 15
TOP_N_BIGRAMAS = 20
FREQ_MINIMA_BIGRAMA = 2

# Incerteza (bootstrap por unidades). Reservado às análises com intervalo de confiança,
# que só entram por comando do pesquisador (skill 02, seção 4.2, item 6).
N_BOOTSTRAP = 2000
NIVEL_CONFIANCA = 0.95
SEMENTE = 2026

## 8. Ambiente, estilo e funções

Restrição de pasta, pasta de saída (usada apenas quando o pesquisador ordenar a gravação em `resultados/`), padrão gráfico e funções auxiliares.

In [ ]:
# ---------------------------------------------------------------------------
# Restrição de ambiente: o notebook só roda de dentro da pasta "Planos Nacionais de IA".
# ---------------------------------------------------------------------------
PASTA = Path.cwd()
if unicodedata.normalize("NFC", PASTA.name) != "Planos Nacionais de IA":
    raise RuntimeError(
        "Este notebook deve ser executado de dentro da pasta 'Planos Nacionais de IA' "
        f"(restrição de ambiente da skill). Pasta atual: {PASTA}"
    )

# ---------------------------------------------------------------------------
# Pasta de saída versionada (resultados/<SIGLA>/<VERSAO_SAIDA>/): resultados
# anteriores nunca são sobrescritos.
# ---------------------------------------------------------------------------
DIR_SAIDA = PASTA / "resultados" / NOME_SAIDA / VERSAO_SAIDA
if SALVAR_SAIDAS:
    if DIR_SAIDA.exists() and any(DIR_SAIDA.iterdir()) and not SOBRESCREVER:
        raise FileExistsError(
            f"A pasta de saída '{DIR_SAIDA.relative_to(PASTA)}' já contém arquivos. "
            "Para preservar a versão anterior, altere VERSAO_SAIDA (ex.: 'v2') na configuração. "
            "Use SOBRESCREVER = True somente se a substituição for intencional."
        )
    DIR_SAIDA.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Estilo das figuras (padrão acadêmico: fonte >= 9 pt, 300 dpi, PNG + PDF)
# ---------------------------------------------------------------------------
# Cores fixas por plano, iguais em todos os notebooks da pasta (skill 02, seção 3.7).
# Paleta inicial, editável pelo pesquisador. Planos do mesmo país ou bloco recebem
# tons da mesma cor.
COR_DOCUMENTO = {
    "PBIA_2025": "#2E9E4F",    # Brasil (verde)
    "EUA_2025": "#1F3F8F",     # Estados Unidos (azul-escuro)
    "UE_2025": "#EDB120",      # União Europeia (amarelo)
    "China_2025": "#B2182B",   # China (vermelho)
}
COR_TINTA = "#0b0b0b"     # texto principal
COR_TINTA_2 = "#52514e"   # texto secundário (rótulos, anotações)
COR_GRADE = "#e4e3df"     # grade e linhas de apoio

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": COR_TINTA_2,
    "axes.linewidth": 0.6,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": COR_GRADE,
    "grid.linewidth": 0.6,
    "grid.linestyle": "-",
    "text.color": COR_TINTA,
    "axes.labelcolor": COR_TINTA,
    "xtick.color": COR_TINTA_2,
    "ytick.color": COR_TINTA_2,
    "legend.frameon": False,
    "pdf.fonttype": 42,
})

# ---------------------------------------------------------------------------
# Funções auxiliares de formatação e gravação
# ---------------------------------------------------------------------------
def num(x, casas=2):
    """Formata número no padrão brasileiro (vírgula decimal, ponto de milhar)."""
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return "—"
    texto = f"{x:,.{casas}f}"
    return texto.replace(",", "§").replace(".", ",").replace("§", ".")


def num_auto(x):
    """Inteiros sem casas decimais; demais valores com duas casas (padrão brasileiro)."""
    return num(x, 0) if float(x).is_integer() else num(x, 2)


# Rótulos de eixo com vírgula decimal.
FMT_BR = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ","))


def exibir(df, casas=2, titulo=None):
    """Exibe uma tabela com vírgula decimal, sem alterar os valores."""
    estilo = df.style.format(precision=casas, decimal=",", thousands=".", na_rep="—")
    display(estilo.set_caption(titulo) if titulo else estilo)


def salvar_figura(fig, nome):
    """Grava a figura em PNG (300 dpi) e PDF na pasta de saída versionada."""
    if SALVAR_SAIDAS:
        for ext in ("png", "pdf"):
            fig.savefig(DIR_SAIDA / f"{nome}.{ext}")


def salvar_tabela(df, nome, index=True):
    """Grava a tabela completa em CSV (UTF-8 com BOM, ';', vírgula decimal)."""
    if SALVAR_SAIDAS:
        df.to_csv(DIR_SAIDA / f"{nome}.csv", sep=";", decimal=",", encoding="utf-8-sig", index=index)


def tom(cor_hex, fracao_branco):
    """Tom da cor do plano: mistura a cor com branco (0 = cor pura; 1 = branco)."""
    rgb = np.array([int(cor_hex[i:i + 2], 16) for i in (1, 3, 5)], dtype=float)
    mistura = rgb + (255 - rgb) * fracao_branco
    return "#" + "".join(f"{int(round(v)):02x}" for v in mistura)


print(f"Pasta de trabalho: {PASTA}")
print(f"Saídas: {DIR_SAIDA.relative_to(PASTA) if SALVAR_SAIDAS else 'gravação desativada'}")

if SIGLA not in COR_DOCUMENTO:
    raise KeyError(f"Defina a cor do plano '{SIGLA}' em COR_DOCUMENTO (skill 02, seção 3.7).")
COR = COR_DOCUMENTO[SIGLA]
# Tipos de unidade no gráfico de extensão: tons da cor do plano, do mais escuro
# (parágrafos) ao mais claro (títulos), sempre acompanhados de legenda.
COR_TIPO = {"paragraph": COR, "list_item": tom(COR, 0.25), "heading": tom(COR, 0.55), "title": tom(COR, 0.55)}

In [ ]:
# ===========================================================================
# PIPELINE DE PRÉ-PROCESSAMENTO E MEDIDAS
# (idêntico em todos os notebooks de análise da pasta "Planos Nacionais de IA")
# ===========================================================================

# Token (palavra) = sequência de letras, admitindo hífen interno (multi-stakeholder)
# ou sigla com pontos (U.S.). Números, pontuação e símbolos NÃO são tokens.
RE_PALAVRA = re.compile(r"(?:[^\W\d_]\.){2,}|[^\W\d_]+(?:[-‐‑][^\W\d_]+)*")
HIFENS = {"-", "‐", "‑"}

# Tipos de unidade e rótulos usados nas tabelas e figuras. "title" é a unidade 0
# (título e subtítulo do plano); os demais tipos vêm do JSON da skill 01.
TIPOS_JSON = ("heading", "paragraph", "list_item")
ROTULO_TIPO = {
    "title": "Título e subtítulo (unidade 0)",
    "heading": "Títulos internos",
    "paragraph": "Parágrafos",
    "list_item": "Itens de lista",
}
ROTULO_TIPO_SINGULAR = {
    "title": "título e subtítulo",
    "heading": "título interno",
    "paragraph": "parágrafo",
    "list_item": "item de lista",
}


def normalizar_hifen(texto):
    """Uniformiza variantes tipográficas do hífen (apenas nas chaves de contagem;
    o texto do JSON não é alterado)."""
    return texto.replace("‐", "-").replace("‑", "-")


def carregar_extracao(caminho):
    """Lê o JSON produzido pela skill 01 e devolve SOMENTE o título, o subtítulo e as
    unidades (id, tipo, categoria e texto integral). Todo o conteúdo periférico
    (number, level, page, path e demais metadados) é descartado aqui.
    Interrompe a execução se o arquivo não passar nas verificações de integridade."""
    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo '{caminho.name}' não encontrado na pasta. "
            "Execute antes a skill 01 (skill_extração.md) para gerar o JSON do plano."
        )
    bruto = caminho.read_bytes()
    dados = json.loads(bruto.decode("utf-8"))

    problemas = [f"chave '{c}' ausente" for c in ("document", "extraction", "sections") if c not in dados]
    if problemas:
        raise ValueError("Falha na verificação do JSON:\n- " + "\n- ".join(problemas))

    documento, extracao, secoes = dados["document"], dados["extraction"], dados["sections"]
    if extracao.get("skill") != "skill_extração":
        problemas.append("o JSON não registra ter sido produzido pela skill 01 (extraction.skill)")
    if extracao.get("corpus") != "Planos Nacionais de IA":
        problemas.append("o JSON não registra pertencer ao corpus 'Planos Nacionais de IA' (extraction.corpus)")
    if [s.get("id") for s in secoes] != list(range(1, len(secoes) + 1)):
        problemas.append("os ids das unidades não são sequenciais a partir de 1")
    vazias = [s.get("id") for s in secoes if not isinstance(s.get("text"), str) or not s["text"].strip()]
    if vazias:
        problemas.append(f"unidades com texto vazio: {vazias}")
    if extracao.get("units_extracted") != len(secoes):
        problemas.append(
            f"extraction.units_extracted = {extracao.get('units_extracted')}, mas há {len(secoes)} unidades"
        )
    tipos_invalidos = sorted({str(s.get("type")) for s in secoes} - set(TIPOS_JSON))
    if tipos_invalidos:
        problemas.append(f"tipos de unidade não previstos pela skill 01: {tipos_invalidos}")
    if not documento.get("title"):
        problemas.append("título (document.title) ausente")
    if documento.get("language") != IDIOMA_ESPERADO:
        problemas.append(
            f"idioma declarado no JSON: {documento.get('language')!r}; esperado: {IDIOMA_ESPERADO!r}"
        )
    if problemas:
        raise ValueError("Falha na verificação do JSON:\n- " + "\n- ".join(problemas))

    return {
        "arquivo": caminho.name,
        "sha256": hashlib.sha256(bruto).hexdigest(),
        "titulo": documento["title"],
        "subtitulo": documento.get("subtitle"),
        "idioma": documento.get("language"),
        "extracao": {k: extracao.get(k) for k in ("skill", "skill_version", "corpus", "date")},
        # SOMENTE id (posição), tipo e categoria (filtros) e texto integral
        "unidades": [
            {"id": s["id"], "tipo": s["type"], "categoria": s.get("category"), "texto": s["text"]}
            for s in secoes
        ],
    }


def selecionar_unidades(documento):
    """Monta as unidades candidatas à análise e aplica os filtros da configuração
    (skill 02, seção 3.1). Nada é apagado: cada unidade recebe o motivo da exclusão
    (None quando entra na análise), para que o perfil informe o que cada filtro retirou.
    A unidade 0 reúne o título e o subtítulo como SEGMENTOS separados, para que não se
    formem bigramas artificiais entre o fim do título e o início do subtítulo."""
    candidatas = [
        {"id": 0, "tipo": "title", "categoria": None, "texto": texto}
        for texto in (documento["titulo"], documento["subtitulo"]) if texto
    ] + [dict(u) for u in documento["unidades"]]
    desconhecidas = sorted(set(UNIDADES_EXCLUIDAS) - {u["id"] for u in candidatas})
    if desconhecidas:
        raise ValueError(f"UNIDADES_EXCLUIDAS contém ids inexistentes no JSON: {desconhecidas}")
    for u in candidatas:
        if u["tipo"] == "title" and not INCLUIR_TITULO_NA_CONTAGEM:
            u["motivo_exclusao"] = "INCLUIR_TITULO_NA_CONTAGEM"
        elif u["tipo"] == "heading" and not INCLUIR_TITULOS_INTERNOS:
            u["motivo_exclusao"] = "INCLUIR_TITULOS_INTERNOS"
        elif u["categoria"] in CATEGORIAS_EXCLUIDAS:
            u["motivo_exclusao"] = "CATEGORIAS_EXCLUIDAS"
        elif u["id"] in UNIDADES_EXCLUIDAS:
            u["motivo_exclusao"] = "UNIDADES_EXCLUIDAS"
        else:
            u["motivo_exclusao"] = None
    return candidatas


# Possíveis resíduos de extração: número colado a palavra ou pontuação (marcador de
# nota, como "Plan.1"), endereço da internet e unidade formada só por números.
RE_MARCADOR_NOTA = re.compile(r"(?<=[a-z.,;:”’)\]])\d{1,3}(?=[\s,;.:]|$)")
RE_URL = re.compile(r"https?://|www\.", re.IGNORECASE)
RE_SO_NUMEROS = re.compile(r"[\d\s.,;:()\-–—]+")


def examinar_inflacao(unidades, titulo, min_palavras_frase=8):
    """Procura sinais de inflação (skill 02, seção 3.5) nas unidades analisadas e apenas
    os informa; o tratamento, quando cabível, é feito em UNIDADES_EXCLUIDAS.
    (a) unidades com texto idêntico; (b) frases idênticas, com pelo menos
    min_palavras_frase palavras, presentes em mais de uma unidade; (c) possíveis resíduos
    de extração: marcadores de nota colados, URLs, unidades só com números e repetições
    do título do plano fora da unidade 0 (cabeçalho de página)."""
    sinais = []
    chave = lambda t: " ".join(t.split()).casefold()

    por_texto = {}
    for u in unidades:
        por_texto.setdefault(chave(u["texto"]), []).append(u["id"])
    for texto, ids in por_texto.items():
        if len(set(ids)) > 1:
            sinais.append({"sinal": "unidades com texto idêntico", "ids": ids, "trecho": texto[:120]})

    por_frase = {}
    for u in unidades:
        for frase in re.split(r"(?<=[.!?])\s+", u["texto"]):
            if len(frase.split()) >= min_palavras_frase:
                por_frase.setdefault(chave(frase), set()).add(u["id"])
    for frase, ids in por_frase.items():
        if len(ids) > 1 and frase not in por_texto:
            sinais.append({"sinal": "frase idêntica em mais de uma unidade", "ids": sorted(ids), "trecho": frase[:120]})

    for u in unidades:
        texto = u["texto"]
        for m in RE_MARCADOR_NOTA.finditer(texto):
            sinais.append({"sinal": "possível marcador de nota colado", "ids": [u["id"]],
                           "trecho": texto[max(0, m.start() - 30): m.end() + 10]})
        if RE_URL.search(texto):
            sinais.append({"sinal": "URL no texto", "ids": [u["id"]], "trecho": texto[:120]})
        if RE_SO_NUMEROS.fullmatch(texto):
            sinais.append({"sinal": "unidade só com números", "ids": [u["id"]], "trecho": texto})
        if u["tipo"] != "title" and chave(texto) == chave(titulo):
            sinais.append({"sinal": "título repetido (cabeçalho)", "ids": [u["id"]], "trecho": texto})
    return pd.DataFrame(sinais, columns=["sinal", "ids", "trecho"])


def montar_stopwords(nlp):
    """Lista de stopwords do spaCy, ajustada pelas listas editáveis da configuração."""
    base = {w.lower() for w in nlp.Defaults.stop_words}
    return (base | {w.lower() for w in STOPWORDS_ADICIONAIS}) - {w.lower() for w in STOPWORDS_MANTIDAS}


def unir_compostos_hifenizados(doc):
    """Reúne em um único token as palavras compostas com hífen (ex.: multi-stakeholder,
    human-centric, state-of-the-art), que o spaCy separa em vários tokens.
    Só une quando não há espaço em volta do hífen."""
    trechos, i, n = [], 0, len(doc)
    while i < n:
        j = i
        while (j + 2 < n and doc[j].is_alpha and doc[j].whitespace_ == ""
               and doc[j + 1].text in HIFENS and doc[j + 1].whitespace_ == ""
               and doc[j + 2].is_alpha):
            j += 2
        if j > i:
            trechos.append(doc[i:j + 1])
            i = j + 1
        else:
            i += 1
    with doc.retokenize() as rt:
        for trecho in trechos:
            ultimo = trecho[-1]
            # Lema do composto = forma escrita; se o último elemento estiver no plural,
            # usa-se o lema dele (multi-stakeholders -> multi-stakeholder).
            if ultimo.tag_ in {"NNS", "NNPS"}:
                lema = trecho.text[: len(trecho.text) - len(ultimo.text)] + ultimo.lemma_
            else:
                lema = trecho.text
            rt.merge(trecho, attrs={"LEMMA": lema, "POS": ultimo.pos_, "TAG": ultimo.tag_})
    return doc


def processar_documento(unidades, nlp, stopwords):
    """Aplica o pré-processamento a cada unidade (título, título interno, parágrafo ou
    item) e devolve uma tabela com uma linha por token (palavra), na ordem original do
    documento. unidades = lista de (id, texto); um mesmo id pode ter mais de um segmento
    (a unidade 0 reúne título e subtítulo). A numeração das frases continua entre os
    segmentos de uma mesma unidade, de modo que nenhum bigrama atravessa segmentos."""
    linhas, posicao, frases_por_unidade = [], 0, {}
    for (uid, _), doc in zip(unidades, nlp.pipe([texto for _, texto in unidades])):
        doc = unir_compostos_hifenizados(doc)
        for frase in doc.sents:
            k = frases_por_unidade.get(uid, 0)
            frases_por_unidade[uid] = k + 1
            for tok in frase:
                if not RE_PALAVRA.fullmatch(tok.text):
                    continue
                lema_spacy = normalizar_hifen(tok.lemma_.lower())
                linhas.append({
                    "unidade": uid,
                    "frase": f"{uid}.{k}",
                    "i": tok.i,                      # índice no segmento (detecta adjacência)
                    "posicao": posicao,              # posição no documento inteiro
                    "forma": normalizar_hifen(tok.text.lower()),
                    "lema_spacy": lema_spacy,        # lema do spaCy, antes das correções
                    "lema": CORRECOES_DE_LEMA.get(lema_spacy, lema_spacy),
                    "classe": tok.pos_,
                    "etiqueta": tok.tag_,
                })
                posicao += 1
    tokens = pd.DataFrame(linhas)
    tokens["stopword"] = tokens["lema"].isin(stopwords)    # verificada pelo lema
    tokens["conteudo"] = (
        tokens["classe"].isin(CLASSES_DE_CONTEUDO)
        & ~tokens["stopword"]
        & (tokens["lema"].str.len() >= TAMANHO_MINIMO_LEMA)
        & tokens["lema"].map(lambda lema: bool(RE_PALAVRA.fullmatch(lema)))
    )
    return tokens


def perfil_documento(tokens, n_unidades, rotulo):
    """Indicadores descritivos de extensão e vocabulário."""
    conteudo = tokens[tokens["conteudo"]]
    tam = tokens.groupby("unidade").size()
    freq_cont = conteudo["lema"].value_counts()
    # Tokens de classes de conteúdo que não entram na contagem, por motivo
    fora = tokens["classe"].isin(CLASSES_DE_CONTEUDO) & ~tokens["conteudo"]
    curto = tokens["lema"].str.len() < TAMANHO_MINIMO_LEMA
    return pd.Series({
        "Unidades analisadas (título, títulos internos, parágrafos e itens)": n_unidades,
        "Unidades sem nenhum token": n_unidades - tokens["unidade"].nunique(),
        "Tokens (palavras)": len(tokens),
        "Formas distintas": tokens["forma"].nunique(),
        "Tokens de conteúdo": len(conteudo),
        "Tokens de conteúdo (% dos tokens)": len(conteudo) / len(tokens) * 100,
        "Lemas de conteúdo distintos": conteudo["lema"].nunique(),
        "Lemas de conteúdo com uma única ocorrência": int((freq_cont == 1).sum()),
        "Tokens de classes de conteúdo retirados pela lista de stopwords": int((fora & tokens["stopword"]).sum()),
        "Tokens de classes de conteúdo retirados por lema curto (< TAMANHO_MINIMO_LEMA)":
            int((fora & ~tokens["stopword"] & curto).sum()),
        "Tokens de classes de conteúdo retirados por lema fora do padrão de palavra":
            int((fora & ~tokens["stopword"] & ~curto).sum()),
        "Tokens por unidade: média": tam.mean(),
        "Tokens por unidade: mediana": tam.median(),
        "Tokens por unidade: mínimo": tam.min(),
        "Tokens por unidade: máximo": tam.max(),
    }, name=rotulo)


def tabela_recorte(candidatas, tokens_todos):
    """Efeito de cada filtro da configuração (skill 02, seção 3.1): unidades e tokens
    retirados. Os filtros são aplicados na ordem da tabela; cada unidade é atribuída ao
    primeiro filtro que a retira."""
    tam = tokens_todos.groupby("unidade").size()
    filtros = [
        ("Título e subtítulo (unidade 0)", "INCLUIR_TITULO_NA_CONTAGEM", INCLUIR_TITULO_NA_CONTAGEM),
        ("Títulos internos", "INCLUIR_TITULOS_INTERNOS", INCLUIR_TITULOS_INTERNOS),
        ("Categorias estruturais", "CATEGORIAS_EXCLUIDAS", CATEGORIAS_EXCLUIDAS),
        ("Unidades por id (tratamento de inflação)", "UNIDADES_EXCLUIDAS", UNIDADES_EXCLUIDAS),
    ]
    linhas = []
    for rotulo, parametro, valor in filtros:
        ids = sorted({u["id"] for u in candidatas if u["motivo_exclusao"] == parametro})
        linhas.append({
            "filtro": rotulo,
            "configuracao": f"{parametro} = {valor!r}",
            "unidades_retiradas": len(ids),
            "tokens_retirados": int(tam.reindex(ids, fill_value=0).sum()),
        })
    tab = pd.DataFrame(linhas).set_index("filtro")
    tab.loc["Total retirado"] = ["", tab["unidades_retiradas"].sum(), tab["tokens_retirados"].sum()]
    return tab


def lemas_descartados_por_stopword(tokens):
    """Lemas de classes de conteúdo que ficaram fora da contagem por estarem na lista de
    stopwords (skill 02, seção 4.2, item 5), com as suas frequências.
    Ordenação: frequência decrescente; empate -> ordem alfabética."""
    sub = tokens[tokens["classe"].isin(CLASSES_DE_CONTEUDO) & tokens["stopword"]]
    colunas = ["frequencia", "por_1000_tokens", "classes", "formas_observadas"]
    if sub.empty:
        return pd.DataFrame(columns=colunas).rename_axis("lema")
    grupos = sub.groupby("lema")
    tab = pd.DataFrame({
        "frequencia": grupos.size(),
        "por_1000_tokens": grupos.size() / len(tokens) * 1000,
        "classes": grupos["classe"].agg(lambda s: "; ".join(f"{c} ({n})" for c, n in s.value_counts().items())),
        "formas_observadas": grupos["forma"].agg(
            lambda s: "; ".join(f"{f} ({n})" for f, n in s.value_counts().items())
        ),
    })
    tab = tab.reset_index().sort_values(["frequencia", "lema"], ascending=[False, True])
    return tab.set_index("lema")


def tabela_composicao(tam_unidades, tipo_por_unidade):
    """Unidades e tokens por tipo de unidade (título, títulos internos, parágrafos e itens)."""
    df = pd.DataFrame({"tokens": tam_unidades, "tipo": [tipo_por_unidade[u] for u in tam_unidades.index]})
    g = df.groupby("tipo")["tokens"]
    tab = pd.DataFrame({
        "unidades": g.size(),
        "tokens": g.sum(),
        "pct_dos_tokens": g.sum() / df["tokens"].sum() * 100,
        "media_por_unidade": g.mean(),
        "mediana_por_unidade": g.median(),
        "minimo": g.min(),
        "maximo": g.max(),
    })
    tab = tab.reindex([t for t in ROTULO_TIPO if t in tab.index])
    tab.index = [ROTULO_TIPO[t] for t in tab.index]
    tab.loc["Total"] = [len(df), df["tokens"].sum(), 100.0, df["tokens"].mean(),
                        df["tokens"].median(), df["tokens"].min(), df["tokens"].max()]
    return tab.rename_axis("tipo de unidade")


def tabela_frequencias(tokens):
    """Frequência, taxa por 1.000 tokens, alcance e dispersão (DP normalizado) de
    TODOS os lemas de conteúdo. Ordenação: frequência decrescente; empate -> ordem alfabética."""
    n_tokens = len(tokens)
    unidades = sorted(tokens["unidade"].unique())
    proporcao_unidade = (tokens.groupby("unidade").size().reindex(unidades) / n_tokens).to_numpy()
    conteudo = tokens[tokens["conteudo"]]
    cruz = pd.crosstab(conteudo["lema"], conteudo["unidade"]).reindex(columns=unidades, fill_value=0)
    freq = cruz.sum(axis=1)
    # DP de Gries (2008), normalizado por Lijffijt & Gries (2012):
    # 0 = distribuição proporcional ao tamanho das unidades; 1 = concentração máxima.
    v = cruz.div(freq, axis=0).to_numpy()
    dp = 0.5 * np.abs(v - proporcao_unidade).sum(axis=1)
    dp_norm = dp / (1 - proporcao_unidade.min()) if len(unidades) > 1 else np.full(len(freq), np.nan)
    formas = conteudo.groupby("lema")["forma"].agg(
        lambda s: "; ".join(f"{forma} ({n})" for forma, n in s.value_counts().items())
    )
    tab = pd.DataFrame({
        "frequencia": freq,
        "por_1000_tokens": freq / n_tokens * 1000,
        "unidades_com_ocorrencia": (cruz > 0).sum(axis=1),
        "pct_unidades": (cruz > 0).sum(axis=1) / len(unidades) * 100,
        "DP_norm": dp_norm,
        "formas_observadas": formas,
    })
    tab.index.name = "lema"
    tab = tab.reset_index().sort_values(["frequencia", "lema"], ascending=[False, True])
    return tab.set_index("lema")


def relatar_empates(tab, coluna, n, rotulo):
    """Informa termos empatados no ponto de corte de um recorte 'top N'."""
    if len(tab) <= n:
        return
    corte = tab[coluna].iloc[n - 1]
    fora = tab.iloc[n:]
    fora = fora[fora[coluna] == corte]
    if len(fora):
        lista = ", ".join(fora.index[:20]) + (" …" if len(fora) > 20 else "")
        print(
            f"Empate no ponto de corte: o {n}º {rotulo} tem valor {num(corte, 2)}; "
            f"outros {len(fora)} com o mesmo valor ficaram fora do recorte "
            f"(desempate por ordem alfabética): {lista}"
        )


def tabela_bigramas(tokens):
    """Pares de tokens de conteúdo ADJACENTES no texto original (mesma frase, sem
    pontuação entre eles). Não se formam pares artificiais pela remoção de stopwords."""
    a = tokens.iloc[:-1].reset_index(drop=True)
    b = tokens.iloc[1:].reset_index(drop=True)
    ok = (
        (a["frase"].to_numpy() == b["frase"].to_numpy())
        & (b["i"].to_numpy() - a["i"].to_numpy() == 1)
        & a["conteudo"].to_numpy() & b["conteudo"].to_numpy()
    )
    pares = pd.DataFrame({
        "bigrama": (a["lema"] + " " + b["lema"])[ok],
        "forma": (a["forma"] + " " + b["forma"])[ok],
        "unidade": a["unidade"][ok],
    })
    if pares.empty:
        return pd.DataFrame(columns=["frequencia", "por_1000_tokens", "unidades_com_ocorrencia",
                                     "forma_principal", "formas_observadas"]).rename_axis("bigrama")
    grupos = pares.groupby("bigrama")
    tab = pd.DataFrame({
        "frequencia": grupos.size(),
        "por_1000_tokens": grupos.size() / len(tokens) * 1000,
        "unidades_com_ocorrencia": grupos["unidade"].nunique(),
        "forma_principal": grupos["forma"].agg(lambda s: s.value_counts().index[0]),
        "formas_observadas": grupos["forma"].agg(
            lambda s: "; ".join(f"{f} ({n})" for f, n in s.value_counts().items())
        ),
    })
    tab = tab.reset_index().sort_values(["frequencia", "bigrama"], ascending=[False, True])
    return tab.set_index("bigrama")


def grafico_dispersao_posicional(ax, posicoes_por_linha, rotulos, inicios_unidades, cor):
    """Gráfico de dispersão posicional: cada traço é uma ocorrência, na posição
    relativa em que aparece no documento (0 = início; 1 = fim)."""
    if len(inicios_unidades) <= 120:
        for x in inicios_unidades:
            ax.axvline(x, color=COR_GRADE, lw=0.5, zorder=0)
    for y, posicoes in enumerate(posicoes_por_linha):
        ax.vlines(posicoes, y - 0.36, y + 0.36, color=cor, lw=0.9)
    ax.set_yticks(range(len(rotulos)), labels=rotulos)
    ax.set_ylim(len(rotulos) - 0.5, -0.5)
    ax.set_xlim(0, 1)
    ax.grid(False)
    ax.xaxis.set_major_formatter(FMT_BR)
    ax.set_xlabel("Posição relativa no documento (0 = início; 1 = fim)")


def versoes_ambiente(nlp):
    return {
        "python": platform.python_version(),
        "spacy": spacy.__version__,
        "modelo_spacy": f"{MODELO_SPACY} {nlp.meta.get('version', '?')}",
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "matplotlib": matplotlib.__version__,
    }

## 9. Carregamento e verificação do insumo

O JSON da skill 01 é lido em modo somente leitura. A execução é interrompida se: o arquivo não existir; não tiver sido produzido pela skill 01 para o corpus *Planos Nacionais de IA* (`extraction.skill` e `extraction.corpus`); os identificadores não forem sequenciais; houver unidade vazia ou de tipo não previsto; o número de unidades divergir do registrado na extração; faltar o título; ou o idioma não for `en`. O SHA-256 do JSON é exibido.

Em seguida, aplicam-se os filtros da configuração e examinam-se as unidades selecionadas em busca de **sinais de inflação** (skill 02, seção 3.5): unidades com texto idêntico, frases idênticas (com 8 palavras ou mais) em mais de uma unidade, marcadores de nota colados a palavras, URLs, unidades só com números e repetições do título como cabeçalho. O exame apenas informa. O tratamento, quando cabível, é feito em `UNIDADES_EXCLUIDAS`, com o motivo registrado.

In [ ]:
documento = carregar_extracao(PASTA / ARQUIVO_JSON)
candidatas = selecionar_unidades(documento)
analisadas = [u for u in candidatas if u["motivo_exclusao"] is None]
IDS_ANALISE = list(dict.fromkeys(u["id"] for u in analisadas))   # ordem do documento
TIPO_POR_UNIDADE = {u["id"]: u["tipo"] for u in candidatas}

origem = documento["extracao"]
tipos_json = pd.Series([u["tipo"] for u in documento["unidades"]]).value_counts()
categorias_json = pd.Series([u["categoria"] for u in documento["unidades"]]).value_counts()

print(f"Título: {documento['titulo']}")
print(f"Subtítulo: {documento['subtitulo'] or '—'}")
print(f"Arquivo: {documento['arquivo']} (somente leitura)")
print(f"SHA-256: {documento['sha256']}")
print(f"Idioma: {documento['idioma']}")
print(f"Origem: {origem['skill']} {origem['skill_version']} ({origem['date']}), corpus '{origem['corpus']}'")
print(f"Unidades no JSON: {len(documento['unidades'])} ("
      + "; ".join(f"{ROTULO_TIPO[t].lower()}: {tipos_json.get(t, 0)}" for t in TIPOS_JSON) + ")")
print("Categorias estruturais no JSON: " + "; ".join(f"{c}: {n}" for c, n in categorias_json.items()))
print()
print(f"Título e subtítulo nas contagens: {'sim (unidade 0)' if INCLUIR_TITULO_NA_CONTAGEM else 'não'}")
print(f"Títulos internos incluídos: {'sim' if INCLUIR_TITULOS_INTERNOS else 'não'}")
print(f"Categorias excluídas: {', '.join(CATEGORIAS_EXCLUIDAS) if CATEGORIAS_EXCLUIDAS else 'nenhuma'}")
print("Unidades excluídas: "
      + ("; ".join(f"id {i} ({m})" for i, m in UNIDADES_EXCLUIDAS.items()) if UNIDADES_EXCLUIDAS else "nenhuma"))
print(f"Unidades analisadas: {len(IDS_ANALISE)}")

sinais = examinar_inflacao(analisadas, documento["titulo"])
print("\nExame de inflação (skill 02, seção 3.5): ", end="")
if sinais.empty:
    print("nenhum sinal encontrado (unidades idênticas, frases idênticas em mais de uma unidade, "
          "marcadores de nota colados, URLs, unidades só com números ou título repetido como cabeçalho).")
else:
    print(f"{len(sinais)} sinal(is) para conferência; o tratamento, quando cabível, é feito em UNIDADES_EXCLUIDAS.")
    exibir(sinais, titulo="Sinais de inflação para conferência")

## 10. Pré-processamento e perfil do documento

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
nlp = spacy.load(MODELO_SPACY)
STOPWORDS = montar_stopwords(nlp)
# Todas as unidades candidatas são processadas, para que o perfil informe também os
# tokens retirados pelos filtros; a análise usa apenas as unidades selecionadas.
tokens_todos = processar_documento([(u["id"], u["texto"]) for u in candidatas], nlp, STOPWORDS)
tokens = tokens_todos[tokens_todos["unidade"].isin(IDS_ANALISE)].reset_index(drop=True)
tokens["posicao"] = np.arange(len(tokens))          # posição no texto analisado
N_TOKENS = len(tokens)
N_UNIDADES = tokens["unidade"].nunique()

perfil = perfil_documento(tokens, len(IDS_ANALISE), SIGLA).to_frame()
display(perfil.style.format(num_auto).set_caption("Perfil do documento"))
recorte = tabela_recorte(candidatas, tokens_todos)
exibir(recorte, 0, "Recorte do conteúdo: o que cada filtro retirou")
descartados = lemas_descartados_por_stopword(tokens)
exibir(descartados, 2, "Lemas de classes de conteúdo retirados pela lista de stopwords")

salvar_tabela(perfil, "00_perfil_documento")
salvar_tabela(recorte, "00_recorte_conteudo")
salvar_tabela(descartados, "00_lemas_retirados_stopwords")
salvar_tabela(tokens, "00_tokens_processados", index=False)

corrigidos = tokens[tokens["lema_spacy"] != tokens["lema"]]
contagem_correcoes = corrigidos.groupby(["lema_spacy", "lema"]).size()
print(f"Stopwords aplicadas: {len(STOPWORDS)} (lista do spaCy, {len(STOPWORDS_ADICIONAIS)} adicionada(s), "
      f"{len(STOPWORDS_MANTIDAS)} mantida(s))")
print(f"Correções de lema aplicadas: {len(corrigidos)} ocorrência(s), "
      f"{len(contagem_correcoes)} de {len(CORRECOES_DE_LEMA)} regra(s) com efeito: "
      + "; ".join(f"{a} → {b} ({n})" for (a, b), n in contagem_correcoes.items()))
print("Ambiente: " + ", ".join(f"{k} {v}" for k, v in versoes_ambiente(nlp).items()))

**Método e escolhas.** Cada unidade é processada com o modelo `en_core_web_sm` do spaCy. Palavras compostas com hífen (*open-source*, *AI-related*) são reunidas em um só token. **Token** é toda palavra formada por letras (com hífen interno ou sigla com pontos, como *U.S.*); números e pontuação não são tokens. O total de tokens é o **denominador** de todas as taxas por 1.000 tokens. **Tokens de conteúdo** são substantivos, nomes próprios, verbos, adjetivos e advérbios fora da lista de *stopwords* do spaCy, verificada pelo **lema**, para que todas as flexões de uma palavra recebam o mesmo tratamento. O título e o subtítulo formam a unidade 0 e são processados como segmentos separados, para que não se forme um bigrama entre o fim do título e o início do subtítulo. A tabela *Recorte do conteúdo* mostra o efeito de cada filtro da configuração. A tabela de lemas retirados mostra, com as frequências, as palavras de classes de conteúdo que ficaram fora da contagem por estarem na lista de *stopwords*.

**Correções de lematização** (`CORRECOES_DE_LEMA`, com o motivo de cada regra registrado na configuração). Além de *datum* → *data*, já prevista no molde, foram registradas duas classes de correção, identificadas no exame do texto processado, antes do cálculo das frequências: (a) *medium* → *media*, porque o spaCy reduz o plural latino *media* (meios de comunicação) a *medium*, palavra que o plano não usa isoladamente; (b) doze formas dos **títulos internos**, escritos com iniciais maiúsculas (*Title Case*), que o spaCy etiqueta como nome próprio e deixa sem lematização (*Risks*, *Workers*, *Technologies*, *Existing* etc.), enquanto as mesmas palavras são lematizadas normalmente no corpo do texto. Sem essa correção, um mesmo termo ficaria dividido em dois lemas (*risks* nos títulos e *risk* no corpo). A linha *Correções de lema aplicadas*, acima, informa quantas ocorrências cada regra alterou. A tabela de tokens processados (`00_tokens_processados.csv`, exportada para `resultados/` apenas por comando do pesquisador) traz, para cada token, a forma, o lema do spaCy, o lema final, a classe gramatical e a decisão de inclusão.

**Como ler.** O perfil descreve a extensão do documento e do seu vocabulário e serve de referência para dimensionar as frequências apresentadas a seguir. As duas tabelas seguintes mostram o que ficou fora da contagem: as unidades e os tokens retirados pelos filtros e os lemas de classes de conteúdo retirados pela lista de *stopwords*.

**Interpretação.** *Observação.* O texto analisado tem 8.198 tokens em 184 unidades: o título e o subtítulo (unidade 0), 34 títulos internos, 47 parágrafos e 102 itens de lista. Nenhum filtro retirou unidades, e o exame de inflação não encontrou sinais. Os tokens de conteúdo são 5.067 (61,81% dos tokens), distribuídos em 1.453 lemas distintos, dos quais 766 ocorrem uma única vez. A lista de *stopwords* retirou 260 tokens de classes de conteúdo; os lemas mais frequentes entre eles são *other* (33), *also* (18), *have* (16), *make* (14) e *such* (14). As 14 regras de correção de lema alteraram 44 ocorrências, 25 delas pela regra *datum* → *data*. *Inferência.* O núcleo de termos repetidos, descrito nas seções seguintes, convive com um vocabulário de cauda longa: mais da metade dos lemas de conteúdo (766 de 1.453) aparece uma única vez. Essa proporção depende da extensão do texto e não deve ser lida, sozinha, como indicador de riqueza vocabular.

**Fragilidades e ajustes possíveis.** O lematizador e o etiquetador do spaCy cometem erros, sobretudo em siglas, nomes próprios e termos técnicos. Por isso, as tabelas trazem as formas observadas de cada lema, e as correções conhecidas ficam explícitas em `CORRECOES_DE_LEMA`. O spaCy não lematiza nomes próprios no plural: *Americans* permanece *americans*, separado de *american*, e *United States* gera *states*, separado de *state*. Essa característica foi mantida, porque lematizar nomes próprios alteraria nomes de instituições (em *Securities and Exchange Commission*, *Securities* viraria *security*). A lista de *stopwords* do spaCy inclui palavras como *may*, *must*, *should* e *will*, que ficam fora da contagem de conteúdo (os modais são tratados à parte, quando o pesquisador pedir), e também verbos e advérbios com algum conteúdo, como *make*, *take* e *also*. A tabela de lemas retirados permite avaliar se algum deles deve voltar à contagem (`STOPWORDS_MANTIDAS`), por decisão do pesquisador.

## 11. Extensão das unidades textuais

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
tam_unidades = tokens.groupby("unidade").size().reindex(IDS_ANALISE, fill_value=0)
tipos = [TIPO_POR_UNIDADE[u] for u in tam_unidades.index]

fig, ax = plt.subplots(figsize=(7.5, 3.4))
x = np.arange(len(tam_unidades))
ax.bar(x, tam_unidades.to_numpy(), width=0.8, color=[COR_TIPO[t] for t in tipos])
ax.axhline(tam_unidades.mean(), color=COR_TINTA_2, lw=0.8)
# Valor da média na margem direita, fora da área das barras.
ax.text(1.01, tam_unidades.mean(), f"média\n{num(tam_unidades.mean(), 1)}", transform=ax.get_yaxis_transform(),
        ha="left", va="center", fontsize=9, color=COR_TINTA_2)
passo = max(1, int(np.ceil(len(x) / 20)))
ax.set_xticks(x[::passo], labels=[str(u) for u in tam_unidades.index[::passo]])
ax.set_xlim(-0.6, len(x) - 0.4)
ax.set_xlabel("Unidade (id no JSON; 0 = título e subtítulo), na ordem do documento")
ax.set_ylabel("Tokens (palavras)")
ax.set_title(f"{SIGLA} · extensão das unidades textuais", loc="left")
ax.grid(axis="x", visible=False)
ax.yaxis.set_major_formatter(FMT_BR)
legenda = [("paragraph", "Parágrafos"), ("list_item", "Itens de lista"),
           ("heading", "Título do plano e títulos internos")]
ax.legend(handles=[Patch(facecolor=COR_TIPO[t], label=r) for t, r in legenda
                   if t in tipos or (t == "heading" and "title" in tipos)],
          loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, handlelength=1.2, columnspacing=1.5)
salvar_figura(fig, "01_extensao_unidades")
plt.show()

composicao = tabela_composicao(tam_unidades, TIPO_POR_UNIDADE)
display(composicao.style.format(num_auto).set_caption("Extensão por tipo de unidade (tokens)"))
salvar_tabela(tam_unidades.rename("tokens").to_frame().assign(tipo=tipos), "01_extensao_unidades")
salvar_tabela(composicao, "01_extensao_por_tipo")

maior = tam_unidades[tam_unidades == tam_unidades.max()]
menor = tam_unidades[tam_unidades == tam_unidades.min()]
descr = lambda serie: ", ".join(f"id {u} ({ROTULO_TIPO_SINGULAR[TIPO_POR_UNIDADE[u]]})" for u in serie.index)
print(f"Síntese factual: {len(tam_unidades)} unidades; média de {num(tam_unidades.mean(), 1)} tokens "
      f"(desvio-padrão {num(tam_unidades.std(), 1)}; mediana {num(tam_unidades.median(), 0)}). "
      f"Maior extensão: {tam_unidades.max()} token(s), em {descr(maior)}. "
      f"Menor extensão: {tam_unidades.min()} token(s), em {len(menor)} unidade(s): {descr(menor)}. "
      "Por tipo: " + "; ".join(f"{ROTULO_TIPO[t].lower()}, média de {num(composicao.loc[ROTULO_TIPO[t], 'media_por_unidade'], 1)}"
                              for t in ("paragraph", "list_item", "heading") if ROTULO_TIPO[t] in composicao.index) + ".")

**Método e escolhas.** Número de tokens (palavras) de cada unidade, na ordem original do documento: a unidade 0 (título e subtítulo), seguida das unidades do JSON (títulos internos, parágrafos e itens de lista). A linha horizontal marca a média de todas as unidades, com o valor indicado à direita do gráfico. Nenhuma unidade é omitida. Em relação ao molde, acrescentou-se a distinção do **tipo de unidade** por tons da cor do plano: parágrafos no tom mais escuro, itens de lista no tom intermediário, título do plano e títulos internos no tom mais claro. A distinção foi acrescentada porque, nesta pasta, os títulos internos entram na análise (skill 02, seção 3.1) e, por serem curtos, puxam a média para baixo. A tabela *Extensão por tipo de unidade* traz os valores de cada tipo.

**Como ler.** Cada barra é uma unidade. Barras altas indicam unidades extensas, e o tom indica o tipo. A sequência mostra como o texto se organiza: cada seção começa por um título interno (barra clara e curta), seguido de parágrafos e de itens de lista.

**Interpretação.** *Observação.* As 184 unidades têm, em média, 44,6 tokens (desvio-padrão 29,4; mediana 41), mas a média reúne tipos de extensão muito diferente. Os parágrafos têm média de 72,7 tokens (47 unidades; 41,69% dos tokens); os itens de lista, 44,6 (102 unidades; 55,48% dos tokens); os títulos internos, 6,6 (34 unidades; 2,74% dos tokens). A maior unidade é o parágrafo de id 41 (141 tokens), o primeiro depois do título interno *Empower American Workers in the Age of AI* (id 40). A menor é o título interno de id 1, *Introduction* (1 token). No início do gráfico, entre os ids 2 e 11, há apenas barras escuras (parágrafos), quatro delas acima de 85 tokens. A partir do id 12, as barras alternam o tom claro e curto dos títulos internos, o escuro dos parágrafos e o intermediário dos itens de lista. *Inferência.* Mais da metade do texto (55,48% dos tokens) está em itens de lista, o que sugere um plano organizado sobretudo como enumeração de medidas, precedida, em cada seção, por parágrafos de contextualização. A Introduction (ids 1 a 11) é o único trecho extenso formado só por parágrafos. Essa forma tem efeito sobre as frequências: fórmulas que se repetem no início dos itens pesam na contagem (seção 12).

**Fragilidades e ajustes possíveis.** A segmentação em unidades é a da skill 01, que segue a estrutura do original. Unidades muito curtas, como os títulos internos, tornam instáveis as medidas por unidade (alcance e dispersão), o que deve ser lembrado na leitura das seções seguintes. A média de todas as unidades mistura tipos de extensão muito diferente; para descrever parágrafos e itens, a tabela por tipo é mais informativa. Os tons de uma mesma cor distinguem-se pela luminosidade, e não pelo matiz; por isso, a legenda acompanha o gráfico, e os valores por tipo estão na tabela.

## 12. Lemas de conteúdo mais frequentes

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
freq = tabela_frequencias(tokens)
salvar_tabela(freq, "02_frequencias_lemas_completa")
relatar_empates(freq, "frequencia", TOP_N_LEMAS, "lema")
top = freq.head(TOP_N_LEMAS)

fig, ax = plt.subplots(figsize=(7.5, 0.27 * len(top) + 1.3))
y = np.arange(len(top))
ax.barh(y, top["frequencia"], height=0.7, color=COR)
ax.set_yticks(y, labels=top.index)
ax.invert_yaxis()
xmax = top["frequencia"].max()
for yi, (_, linha) in zip(y, top.iterrows()):
    ax.text(linha["frequencia"] + xmax * 0.015, yi,
            f"{num(linha['por_1000_tokens'])} por 1.000 · {int(linha['unidades_com_ocorrencia'])}/{N_UNIDADES} unid.",
            va="center", fontsize=9, color=COR_TINTA_2)
ax.set_xlim(0, xmax * 1.55)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.xaxis.set_major_formatter(FMT_BR)
ax.set_xlabel("Frequência absoluta (ocorrências)")
ax.set_title(f"{SIGLA} · {len(top)} lemas de conteúdo mais frequentes", loc="left")
ax.grid(axis="y", visible=False)
salvar_figura(fig, "02_lemas_mais_frequentes")
plt.show()

exibir(top[["frequencia", "por_1000_tokens", "unidades_com_ocorrencia", "pct_unidades", "DP_norm", "formas_observadas"]])
n_conteudo = int(tokens["conteudo"].sum())
print(f"Síntese factual: os {len(top)} lemas acima somam {num(top['frequencia'].sum(), 0)} ocorrências, "
      f"{num(top['frequencia'].sum() / n_conteudo * 100, 1)}% dos {num(n_conteudo, 0)} tokens de conteúdo. "
      f"O documento tem {num(len(freq), 0)} lemas de conteúdo distintos (tabela completa exportada apenas por comando do pesquisador).")

**Método e escolhas.** Frequência absoluta dos *N* lemas de conteúdo mais frequentes (*N* = `TOP_N_LEMAS`). O rótulo de cada barra traz a frequência por 1.000 tokens e o **alcance** (em quantas unidades o lema aparece). A tabela acrescenta o **DP normalizado** (dispersão) e as formas observadas. Ordenação: frequência decrescente; empates resolvidos por ordem alfabética e informados acima do gráfico quando afetam o corte. A tabela completa de todos os lemas (`02_frequencias_lemas_completa.csv`) é exportada para `resultados/` apenas por comando do pesquisador.

**Como ler.** Barras mais longas indicam lemas mais frequentes. Um lema frequente presente em muitas unidades é estruturante no texto; um lema frequente presente em poucas unidades é localizado. A distinção aparece no rótulo (alcance) e na coluna DP_norm da tabela (0 = espalhado; 1 = concentrado).

**Interpretação.** *Observação.* *ai* é de longe o lema mais frequente: 244 ocorrências (29,76 por 1.000 tokens), presente em 136 das 184 unidades (73,91%), com o menor DP do recorte (0,30). O segundo, *lead* (59 ocorrências; 7,20 por 1.000), está em 58 unidades, quase uma ocorrência por unidade, e 46 das suas ocorrências vêm da forma *led*, que no texto aparece na construção *Led by* (ou *led by*), com que as medidas indicam o órgão responsável. Seguem *system* (52), *doc* e *federal* (49 cada), *new* (42), *national* (40), *agency* e *use* (39 cada). Os 25 lemas do recorte somam 1.106 ocorrências, 21,8% dos 5.067 tokens de conteúdo. *doc* é a sigla do *Department of Commerce*, assim definida no texto (*Department of Commerce (DOC)*), e é a única sigla de órgão entre os 25 lemas. As referências ao próprio país aparecem em três lemas do recorte: *american* (35), *america* (26) e *u.s.* (25). *control* tem o DP mais alto do recorte (0,92; 26 ocorrências em 16 unidades); 12 delas pertencem ao bigrama *export control(s)* (seção 14). *Inferência.* O recorte reúne três conjuntos de termos: (a) o aparato do governo federal e a execução das medidas (*lead*, *doc*, *federal*, *agency*, *government*, *program*, *ensure*); (b) a tecnologia e a sua base material (*system*, *model*, *data*, *technology*, *infrastructure*, *research*, *development*, *build*); (c) a referência ao próprio país (*american*, *america*, *u.s.*). O peso de (a), somado à forma de lista vista na seção 11, sugere um texto voltado à atribuição de tarefas a órgãos do Executivo federal. *security* (34) e *national* (40) ficam fora desses conjuntos; a ligação entre eles aparece na expressão *national security* (seção 14).

**Fragilidades e ajustes possíveis.** Frequência não equivale a importância política: um termo pode ser frequente por razões de estilo ou por repetição formular. Lemas genéricos, como o próprio *ai*, tendem a ocupar o topo em planos sobre IA. Isso é um resultado, e não um erro, e por isso esses lemas não são removidos sem decisão do pesquisador. *AI* e *artificial intelligence* não são unificados, porque isso alteraria o texto; se o pesquisador desejar, a unificação pode ser feita por regra explícita na configuração. Neste plano, *lead* deve a maior parte da sua frequência à construção *Led by* (46 das 59 ocorrências vêm da forma *led*). A construção é texto do plano e, por isso, não foi tratada como inflação. Se o pesquisador quiser medir *lead* apenas no sentido de liderança, pode retirar o lema por `STOPWORDS_ADICIONAIS` ou pedir uma contagem sem a construção. As referências ao próprio país estão divididas entre *american*, *america*, *u.s.* e os lemas de *United States* (*united* e *states*); a unificação, se desejada, também é decisão do pesquisador. Siglas de órgãos (*doc*) entram como nomes próprios e somam-se às demais palavras de conteúdo.

## 13. Distribuição dos lemas ao longo do documento

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
lemas_disp = freq.head(TOP_N_DISPERSAO).index.tolist()
inicio_unidade = tokens.groupby("unidade")["posicao"].min() / N_TOKENS
# Linhas de referência: início de cada título interno (pilares e seções). Se os títulos
# internos estiverem fora da análise, usa-se o início de cada unidade, como no molde.
ids_titulos = [u for u in IDS_ANALISE if TIPO_POR_UNIDADE[u] == "heading"]
inicios = (inicio_unidade.reindex(ids_titulos) if ids_titulos else inicio_unidade).dropna().to_numpy()
posicoes = [tokens.loc[tokens["conteudo"] & (tokens["lema"] == lema), "posicao"].to_numpy() / N_TOKENS
            for lema in lemas_disp]
rotulos = [f"{lema}  (DP = {num(freq.loc[lema, 'DP_norm'])})" for lema in lemas_disp]

fig, ax = plt.subplots(figsize=(7.5, 0.33 * len(lemas_disp) + 1.3))
grafico_dispersao_posicional(ax, posicoes, rotulos, inicios, COR)
ax.set_title(f"{SIGLA} · posição das ocorrências dos {len(lemas_disp)} lemas mais frequentes", loc="left")
salvar_figura(fig, "03_dispersao_lemas")
plt.show()

# Ocorrências por quinto do texto (faixas de posição relativa, e não divisões do plano)
QUINTOS = ["0–0,2", "0,2–0,4", "0,4–0,6", "0,6–0,8", "0,8–1"]
por_quinto = pd.DataFrame([np.histogram(p, bins=np.linspace(0, 1, 6))[0] for p in posicoes],
                          index=pd.Index(lemas_disp, name="lema"), columns=QUINTOS)
disp = freq.head(TOP_N_DISPERSAO)[["frequencia", "unidades_com_ocorrencia", "pct_unidades", "DP_norm"]].join(por_quinto)
exibir(disp.sort_values("DP_norm"), titulo="Dispersão e ocorrências por quinto do texto (posição relativa)")
salvar_tabela(disp, "03_dispersao_lemas")

# Linhas de referência do gráfico: posição relativa de início de cada título interno
referencias = pd.DataFrame({
    "posicao_relativa": inicio_unidade.reindex(ids_titulos).to_numpy(),
    "titulo_interno": [next(u["texto"] for u in analisadas if u["id"] == i) for i in ids_titulos],
}, index=pd.Index(ids_titulos, name="id"))
if len(referencias):
    exibir(referencias, 3, "Linhas de referência: início de cada título interno")
print(f"Síntese factual: DP normalizado entre {num(freq.head(TOP_N_DISPERSAO)['DP_norm'].min())} "
      f"e {num(freq.head(TOP_N_DISPERSAO)['DP_norm'].max())} nos {len(lemas_disp)} lemas; "
      f"linhas de referência no início de {len(inicios)} títulos internos.")

**Método e escolhas.** Cada traço vertical é uma ocorrência do lema, posicionada pela sua posição relativa no documento (posição do token ÷ total de tokens). As linhas verticais claras marcam o **início de cada título interno** (pilares e seções). No molde, essas linhas marcavam o início de cada unidade; aqui, com 184 unidades, elas encobririam os traços, e a função do molde as omite acima de 120 unidades. Por isso, usa-se como referência o início dos títulos internos, que delimitam as seções do texto, sem nenhum agrupamento das contagens por pilar ou seção. O DP normalizado (Gries, 2008; Lijffijt & Gries, 2012) resume a distribuição: 0 indica ocorrências proporcionais ao tamanho das unidades; 1, concentração máxima. Em relação ao molde, a tabela acrescenta as **ocorrências por quinto do texto** (cinco faixas iguais de posição relativa: 0–0,2, 0,2–0,4 etc.), que quantificam o que o gráfico mostra, e uma segunda tabela identifica cada **linha de referência** (id, posição relativa e texto do título interno). Os quintos são faixas de posição, e não divisões do plano.

**Como ler.** Linhas com traços espalhados indicam termos presentes em todo o texto; traços agrupados indicam termos concentrados em uma parte (por exemplo, entre duas linhas claras, isto é, dentro de uma seção). A tabela ordena os lemas do mais espalhado ao mais concentrado.

**Interpretação.** *Observação.* Entre os 15 lemas mais frequentes, só *ai* tem DP abaixo de 0,6 (0,30), com ocorrências em todos os quintos do texto (entre 36 e 61 por quinto). *lead* (DP 0,66) está em 58 unidades e cobre todos os quintos (de 8 a 18 ocorrências por quinto), mas não ocorre na Introduction: a primeira ocorrência vem depois do título *Pillar I: Accelerate AI Innovation* (posição 0,079). Os demais têm DP entre 0,71 (*system*) e 0,83 (*data* e *agency*), e vários se concentram em faixas do texto. *infrastructure* tem 18 das 34 ocorrências entre 0,6 e 0,8, faixa situada entre os títulos *Pillar II: Build American AI Infrastructure* (0,567) e *Pillar III: Lead in International AI Diplomacy and Security* (0,835). *security* tem 15 das 34 ocorrências e *doc* 19 das 49 no último quinto (0,8–1), faixa que começa perto dos títulos *Promote Mature Federal Capacity for AI Incident Response* (0,808) e *Pillar III* (0,835). *federal*, ao contrário, tem 42 das 49 ocorrências nos três primeiros quintos e apenas 7 nos dois últimos. *agency* e *use* têm o maior número de ocorrências no quinto central (15 cada), que inclui o título *Accelerate AI Adoption in Government* (0,455). *data* forma dois grupos, com 12 ocorrências entre 0,2 e 0,4 (onde estão *Invest in AI-Enabled Science*, 0,323, e *Build World-Class Scientific Datasets*, 0,353) e 12 entre 0,6 e 0,8 (onde está *Build High-Security Data Centers for Military and Intelligence Community Usage*, 0,690). *new* tem 10 ocorrências no primeiro quinto, 8 delas no parágrafo de id 3, que enumera *new materials*, *new chemicals*, *new drugs* e *new methods*. *Inferência.* Os termos estruturantes do texto inteiro são poucos: *ai* e, pela construção *Led by*, *lead*. Os demais lemas frequentes ligam-se a partes específicas do plano, e a sua frequência total resulta, em boa medida, de seções temáticas, e não de presença uniforme. O caso de *data* sugere dois usos distintos do mesmo lema, os dados científicos e os *data centers*, que a contagem por lema soma (ver o bigrama *data centers*, seção 14).

**Fragilidades e ajustes possíveis.** O DP é sensível à segmentação: uma única unidade longa pode concentrar ocorrências por ser longa, e não por ênfase. O DP também depende da frequência: um lema com menos ocorrências do que o número de unidades (184) não pode estar em todas elas e, por isso, tem DP alto mesmo quando espalhado pelo texto. O DP deve ser lido em termos relativos, entre os lemas deste documento. Os quintos são faixas iguais de posição, e não divisões do plano: uma concentração em um quinto pode atravessar mais de uma seção, e a correspondência com as seções deve ser feita pela tabela das linhas de referência. O gráfico mostra posições, não intensidade.

## 14. Expressões recorrentes (bigramas)

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
bigramas = tabela_bigramas(tokens)
salvar_tabela(bigramas, "04_bigramas_completa")
elegiveis = bigramas[bigramas["frequencia"] >= FREQ_MINIMA_BIGRAMA]
relatar_empates(elegiveis, "frequencia", TOP_N_BIGRAMAS, "bigrama")
topb = elegiveis.head(TOP_N_BIGRAMAS)

if topb.empty:
    print(f"Nenhum bigrama atinge a frequência mínima de {FREQ_MINIMA_BIGRAMA}; gráfico não gerado.")
else:
    fig, ax = plt.subplots(figsize=(7.5, 0.27 * len(topb) + 1.3))
    y = np.arange(len(topb))
    ax.barh(y, topb["frequencia"], height=0.7, color=COR)
    ax.set_yticks(y, labels=topb["forma_principal"])
    ax.invert_yaxis()
    xmax = topb["frequencia"].max()
    for yi, (_, linha) in zip(y, topb.iterrows()):
        ax.text(linha["frequencia"] + xmax * 0.015, yi,
                f"{int(linha['unidades_com_ocorrencia'])}/{N_UNIDADES} unid.",
                va="center", fontsize=9, color=COR_TINTA_2)
    ax.set_xlim(0, xmax * 1.25)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.xaxis.set_major_formatter(FMT_BR)
    ax.set_xlabel("Frequência absoluta (ocorrências)")
    ax.set_title(f"{SIGLA} · {len(topb)} expressões de duas palavras mais frequentes", loc="left")
    ax.grid(axis="y", visible=False)
    salvar_figura(fig, "04_bigramas")
    plt.show()
    exibir(topb)
print(f"Síntese factual: {num(len(bigramas), 0)} bigramas distintos; {num(len(elegiveis), 0)} com frequência >= {FREQ_MINIMA_BIGRAMA}.")

**Método e escolhas.** Pares de tokens de conteúdo **adjacentes no texto original**, na mesma frase e sem pontuação entre eles. Os pares são agrupados pelo lema (ex.: *data center* reúne *data centers*). O rótulo mostra a forma escrita mais frequente. Só entram no gráfico os bigramas com frequência mínima `FREQ_MINIMA_BIGRAMA`. Como a adjacência é verificada no texto original, a remoção de *stopwords* não cria pares artificiais (em *Department of Defense*, não se forma *department defense*).

**Como ler.** As barras mostram as expressões que o texto mais repete, frequentemente nomes de instituições ou de conceitos. O rótulo indica em quantas unidades a expressão aparece.

**Interpretação.** *Observação.* Há 1.712 bigramas distintos, dos quais 196 têm frequência igual ou superior a 2. O mais frequente é *ai systems* (32 ocorrências, em 21 unidades), seguido de *united states* (21), *national security* (17), *federal agencies* (14), *data centers* (13), *export controls* (12) e *semiconductor manufacturing* (11). Seis dos 20 bigramas do gráfico contêm *ai*: *ai systems*, *ai infrastructure* (10), *ai adoption* (9), *ai innovation* (8), *american ai* (7) e *advanced ai* (5). Aparecem também expressões que nomeiam o governo e a autoria política do plano: *federal government* (10), *u.s. government* (8), *trump administration* (7), *action plan* (7) e *executive order* (6). O 20º lugar tem empate em 5 ocorrências, e nove bigramas com esse valor ficaram fora do gráfico por ordem alfabética, entre eles *artificial intelligence*, *frontier ai* e *president trump*. *Inferência.* Os bigramas confirmam, como expressões fixas, os conjuntos vistos nos lemas: a IA tratada como sistema e como infraestrutura física (*ai systems*, *ai infrastructure*, *data centers*, *semiconductor manufacturing*), o vocabulário de segurança e de controle (*national security*, *export controls*, *critical infrastructure*) e o governo federal como executor (*federal agencies*, *federal government*). A forma por extenso *artificial intelligence* tem 5 ocorrências (valor do empate informado acima do gráfico), contra 244 do lema *ai* (seção 12): o texto usa quase sempre a sigla.

**Fragilidades e ajustes possíveis.** Expressões com três ou mais palavras aparecem fragmentadas em bigramas sobrepostos (*semiconductor manufacturing* vem também de *semiconductor manufacturing export controls*). Expressões interrompidas por *stopwords* (*Department of Defense*) não são captadas como bigramas. Nomes próprios compostos (*United States*, *Trump Administration*) contam como bigramas, como qualquer outra sequência de duas palavras de conteúdo. O recorte em 20 expressões deixa de fora, por ordem alfabética, nove bigramas empatados na 20ª posição; a lista completa está na tabela exportável.

## 15. Exclusões de conteúdo por comando do pesquisador

*Visualização construída por comando do pesquisador (2026-10-05) e ajustada por novo comando no mesmo dia (menções ao presidente Trump mantidas).*

> **Comando do pesquisador:** “Retire lead que possua a construção led by e retire também a menção a orgãos e instituições governamentais (Como o DOC por exemplo). Inclua Federal somente se não estiver ligado a nenhuma instituição norte-americana. retire u.s, united states, e outros que mencionem o nome do país. Dos 25 termos de unigrama, coloque para 30. Mantenha os 20 bigramas.”

> **Ajuste do pesquisador (2026-10-05):** “Apenas faça uma pequena modificação deixando questões relacionadas ao presidente Trump, tanto em bigramas quanto em unigramas. FAÇA MODIFICANDO E ALTERANDO OS GRÁFICOS NOVOS E DEIXANDO INTACTO OS GRÁFICOS ANTERIORES.” O ajuste foi aplicado nesta seção e nas seções 16 a 18; as seções 1 a 14 não foram alteradas.

As seções 1 a 14 permanecem como estavam e servem de **base sem exclusões**, para comparação. A partir daqui, o comando é aplicado: esta seção define as exclusões e confere o seu efeito; as seções 16 a 18 refazem, com as exclusões, os lemas mais frequentes (agora 30), a distribuição dos 15 lemas mais frequentes e os 20 bigramas mais frequentes.

As palavras retiradas deixam de ser **tokens de conteúdo**, mas continuam no texto e, portanto, no **total de tokens**, que é o denominador das taxas por 1.000 tokens, como acontece com as *stopwords*. Por isso, as taxas das seções 16 a 18 são diretamente comparáveis às das seções 12 a 14, e a extensão das unidades (seção 11) não muda. Os parâmetros e as funções desta etapa ficam nas células desta seção, e não nas seções 7 e 8, para que as células anteriores permaneçam idênticas.

In [ ]:
# =====================================================================
# EXCLUSÕES DE CONTEÚDO — comando do pesquisador (2026-10-05)
# Estes parâmetros ficam aqui, e não na configuração da seção 7, para que as
# seções 1 a 14 permaneçam idênticas (base sem exclusões, para comparação).
# =====================================================================
TOP_N_LEMAS_EXCLUSOES = 30   # recorte de lemas após as exclusões (seção 12: TOP_N_LEMAS = 25)
# Mantêm-se TOP_N_DISPERSAO = 15 e TOP_N_BIGRAMAS = 20 (seção 7).
TERMOS_CONSULTADOS = ["trump", "race"]   # lemas consultados pelo pesquisador: frequência e posição
                                         # no ranking completo com exclusões (seção 16)

# Expressões retiradas da contagem de conteúdo, por categoria (listas editáveis).
# - A busca percorre a sequência de palavras de cada frase, sem distinção de maiúsculas
#   e ignorando a pontuação; em cada posição, vale a expressão mais longa que coincide.
# - Todas as palavras da ocorrência deixam de ser tokens de conteúdo, exceto as escritas
#   entre colchetes, que fazem parte do padrão, mas permanecem na contagem.
# - As palavras retiradas continuam no total de tokens (denominador), como as stopwords.
EXCLUSOES = {
    "Construção led by": [
        "led by",                  # retira "led"; "by" já está fora da contagem (stopword)
    ],
    "Nome de órgão ou instituição governamental": [
        # Departamentos, por extenso e em sigla
        "Department of Commerce", "DOC",
        "Department of Defense", "DOD", "DOD-Intelligence Community",
        "Department of Energy", "DOE", "DOE National Laboratories",
        "Department of Labor", "DOL",
        "Department of Education", "ED", "Office of Career, Technical, and Adult Education",
        "Department of the Treasury",
        "Department of Homeland Security", "DHS",
        "Department of Justice", "DOJ", "Office of Legal Policy",
        "Department of State", "DOS",
        "Department of Agriculture",   # U.S. Department of Agriculture ("U.S." fica na categoria do país)
        "Department of Interior",
        # Agências, escritórios, conselhos, comissões, bureaus e afins
        "National Institute of Standards and Technology", "NIST",
        "Center for AI Standards and Innovation", "CAISI",
        "Office of Science and Technology Policy", "OSTP",
        "Office of Management and Budget", "OMB",
        "Federal Communications Commission", "FCC",
        "Federal Trade Commission", "FTC",
        "National Science Foundation", "NSF",
        "National Telecommunications and Information Administration", "NTIA",
        "Food and Drug Administration", "FDA",
        "Securities and Exchange Commission", "SEC",
        "Office of the Director of National Intelligence", "ODNI",
        "Intelligence Community", "IC",
        "National Security Council", "NSC",
        "Bureau of Labor Statistics", "BLS",
        "Census Bureau",
        "Bureau of Economic Analysis", "BEA",
        "National Science and Technology Council", "NSTC", "Machine Learning and AI Subcommittee",
        "NIH",
        "Defense Advanced Research Projects Agency",
        "Chief Artificial Intelligence Officer Council", "CAIOC",
        "President’s Management Council", "Chief Data Officer Council", "Chief Information Officer Council",
        "Interagency Council on Statistical Policy", "Chief Human Capital Officer Council",
        "Federal Privacy Council",
        "Office of Personnel Management",
        "General Services Administration", "GSA",
        "High Impact Service Providers",
        "Advisory Committee on Evidence Rules",
        "National Energy Dominance Council", "NEDC",
        "Investment Accelerator",      # United States Investment Accelerator ("United States" fica no país)
        "CHIPS Program Office",
        "Bureau of Industry and Security",
        "Office of the National Cyber Director",
        "Cybersecurity and Infrastructure Security Agency",
        "Economic Diplomacy Action Group",
        "Trade and Development Agency",                    # U.S. Trade and Development Agency
        "Export-Import Bank",
        "International Development Finance Corporation",   # U.S. International Development Finance Corporation
        # Órgãos que o plano propõe criar
        "AI Workforce Research Hub", "Hub",                # "The Hub" retoma o AI Workforce Research Hub
        "AI Information Sharing and Analysis Center", "AI-ISAC",
        # Designações próprias do governo e das Forças Armadas
        # ("Trump Administration" foi retirada desta lista: ver EXPRESSOES_PROTEGIDAS)
        "Biden Administration",
        "Armed Forces",
    ],
    "Designação genérica de órgão ou instituição governamental": [
        "agency", "agencies", "government", "governments", "administration",
        "courts", "law enforcement", "regulators", "establishments", "national laboratories",
    ],
    "Federal ligado a órgão ou instituição": [
        "Federal agency", "Federal agencies", "Federal government", "Federal executive councils",
        "Federal [science] agencies", "Federal partners", "Federal enterprise",
    ],
    "Nome do país": [
        "U.S.", "United States", "America", "American", "Americans",
    ],
}

# Expressões protegidas: entram na busca com prioridade, como as demais, mas nenhuma das
# suas palavras é retirada. Ajuste do pesquisador (2026-10-05): manter as menções ao
# presidente Trump em unigramas e bigramas. Sem a proteção, "administration" sairia de
# "Trump Administration" pela designação genérica.
EXPRESSOES_PROTEGIDAS = [
    "Trump Administration",
]

In [ ]:
# ---------------------------------------------------------------------------
# Funções desta etapa. Ficam nesta célula, e não na célula de funções da seção 8,
# para que as seções 1 a 14 permaneçam idênticas.
# ---------------------------------------------------------------------------
def formas_da_expressao(texto, nlp):
    """Sequência de formas (minúsculas) de uma expressão, com o mesmo pré-processamento
    do texto: tokenização do spaCy, reunião de compostos com hífen e padrão de palavra."""
    doc = unir_compostos_hifenizados(nlp(texto))
    return [normalizar_hifen(t.text.lower()) for t in doc if RE_PALAVRA.fullmatch(t.text)]


def preparar_exclusoes(exclusoes, nlp, protegidas=()):
    """Converte as expressões em padrões de formas, da mais longa para a mais curta.
    Palavras entre colchetes fazem parte do padrão, mas não são retiradas; as expressões
    protegidas são tratadas como inteiramente entre colchetes. Interrompe a execução se uma
    expressão ficar vazia ou repetir outra (as categorias são exclusivas)."""
    entradas, vistos = [], {}
    grupos = dict(exclusoes)
    grupos[CATEGORIA_PROTEGIDA] = [f"[{x}]" for x in protegidas]
    for categoria, expressoes in grupos.items():
        for expressao in expressoes:
            formas, retirar = [], []
            for parte in re.split(r"(\[[^\]]*\])", expressao):
                if not parte.strip():
                    continue
                f = formas_da_expressao(parte.strip("[]"), nlp)
                formas += f
                retirar += [not parte.startswith("[")] * len(f)
            chave = tuple(formas)
            if not chave:
                raise ValueError(f"A expressão '{expressao}' não contém nenhuma palavra.")
            if chave in vistos:
                raise ValueError(f"A expressão '{expressao}' repete '{vistos[chave]}'.")
            vistos[chave] = expressao
            entradas.append({"categoria": categoria, "expressao": expressao,
                             "formas": chave, "retirar": tuple(retirar)})
    return sorted(entradas, key=lambda e: -len(e["formas"]))


def aplicar_exclusoes(tokens, entradas):
    """Marca os tokens retirados da contagem de conteúdo. A busca percorre cada frase da
    esquerda para a direita; em cada posição, vale a expressão mais longa que coincide, e
    cada token é retirado no máximo uma vez. O total de tokens não muda.
    Devolve os tokens (com 'exclusao', 'expressao_excluida' e 'conteudo' recalculado) e o
    número de ocorrências de cada expressão."""
    categoria = pd.Series(None, index=tokens.index, dtype=object)
    expressao = pd.Series(None, index=tokens.index, dtype=object)
    ocorrencias = {e["expressao"]: 0 for e in entradas}
    for _, frase in tokens.groupby("frase", sort=False):
        indices, formas = frase.index.to_list(), frase["forma"].to_list()
        j = 0
        while j < len(formas):
            for e in entradas:
                n = len(e["formas"])
                if tuple(formas[j:j + n]) == e["formas"]:
                    for k in range(n):
                        if e["retirar"][k]:
                            categoria[indices[j + k]] = e["categoria"]
                            expressao[indices[j + k]] = e["expressao"]
                    ocorrencias[e["expressao"]] += 1
                    j += n
                    break
            else:
                j += 1
    resultado = tokens.copy()
    resultado["exclusao"] = categoria
    resultado["expressao_excluida"] = expressao
    resultado["conteudo"] = tokens["conteudo"] & categoria.isna()
    return resultado, ocorrencias


def contexto(tokens, indice, janela=5):
    """Palavras vizinhas de um token, na mesma frase (sem pontuação), para conferência."""
    frase = tokens[tokens["frase"] == tokens.at[indice, "frase"]]
    p = frase.index.get_loc(indice)
    antes = " ".join(frase["forma"].iloc[max(0, p - janela):p])
    depois = " ".join(frase["forma"].iloc[p + 1:p + 1 + janela])
    return f"… {antes} [{frase['forma'].iloc[p]}] {depois} …"


# ---------------------------------------------------------------------------
# Aplicação
# ---------------------------------------------------------------------------
CATEGORIA_PROTEGIDA = "Expressão protegida (mantida na contagem)"
entradas_exclusao = preparar_exclusoes(EXCLUSOES, nlp, EXPRESSOES_PROTEGIDAS)
tokens_excl, ocorrencias_excl = aplicar_exclusoes(tokens, entradas_exclusao)
retirados = tokens_excl[tokens_excl["exclusao"].notna()]
era_conteudo = tokens.loc[retirados.index, "conteudo"]

# 1) Efeito por categoria
por_categoria = pd.DataFrame({
    "expressoes_na_lista": pd.Series({c: len(v) for c, v in EXCLUSOES.items()}),
    "ocorrencias": pd.Series({c: sum(ocorrencias_excl[x] for x in v) for c, v in EXCLUSOES.items()}),
    "tokens_retirados": retirados.groupby("exclusao").size(),
    "tokens_de_conteudo_retirados": retirados[era_conteudo].groupby("exclusao").size(),
}).reindex(list(EXCLUSOES)).fillna(0).astype(int)
por_categoria.loc["Total"] = por_categoria.sum()
por_categoria.index.name = "categoria"
exibir(por_categoria, 0, "Exclusões por categoria")

# 2) Ocorrências de cada expressão da lista (inclusive as sem ocorrência)
por_expressao = pd.DataFrame([{"categoria": c, "expressao": x, "ocorrencias": ocorrencias_excl[x]}
                              for c, v in EXCLUSOES.items() for x in v])
exibir(por_expressao.set_index(["categoria", "expressao"]), 0, "Ocorrências de cada expressão da lista")

# 3) Efeito por lema: frequência sem e com as exclusões
sem = tokens.loc[tokens["conteudo"], "lema"].value_counts()
com = tokens_excl.loc[tokens_excl["conteudo"], "lema"].value_counts().reindex(sem.index, fill_value=0)
efeito = pd.DataFrame({"frequencia_sem_exclusoes": sem, "frequencia_com_exclusoes": com})
efeito["retiradas"] = efeito["frequencia_sem_exclusoes"] - efeito["frequencia_com_exclusoes"]
efeito["categorias"] = retirados[era_conteudo].groupby("lema")["exclusao"].agg(
    lambda s: "; ".join(f"{c} ({n})" for c, n in s.value_counts().items()))
efeito = (efeito[efeito["retiradas"] > 0].rename_axis("lema").reset_index()
          .sort_values(["retiradas", "lema"], ascending=[False, True]).set_index("lema"))
exibir(efeito, 0, "Lemas afetados pelas exclusões")

# 4) Verificação de resíduos: ocorrências mantidas de lemas parcialmente retirados
#    e siglas do texto que permaneceram na contagem
VERIFICAR = ["federal", "lead", "office", "united", "states", "trump", "administration"]
mantidos = tokens_excl[tokens_excl["conteudo"] & tokens_excl["lema"].isin(VERIFICAR)]
residuos = pd.DataFrame({
    "lema": mantidos["lema"],
    "unidade": mantidos["unidade"],
    "contexto": [contexto(tokens_excl, i) for i in mantidos.index],
}).sort_values(["lema", "unidade"], kind="stable")
exibir(residuos.set_index(["lema", "unidade"]), 0, "Ocorrências mantidas de lemas sob verificação")

retiradas_simples = {e["formas"][0] for e in entradas_exclusao if len(e["formas"]) == 1}
siglas = pd.Series([m.group(0) for u in analisadas for m in re.finditer(r"\b[A-Z][A-Z&\-]*[A-Z]\b", u["texto"])]).value_counts()
siglas = pd.DataFrame({"ocorrencias": siglas,
                       "situacao": ["retirada" if s.lower() in retiradas_simples else "mantida" for s in siglas.index]})
mantidas = siglas[siglas["situacao"] == "mantida"]
print("Siglas do texto retiradas (órgãos): "
      + "; ".join(f"{s} ({n})" for s, n in siglas.loc[siglas["situacao"] == "retirada", "ocorrencias"].items()))
print("Siglas do texto fora da lista de exclusões (algumas nem são tokens, como R&D e FAST-41): "
      + "; ".join(f"{s} ({n})" for s, n in mantidas["ocorrencias"].items()))

print("Expressões protegidas (mantidas na contagem por comando do pesquisador): "
      + "; ".join(f"{x} ({ocorrencias_excl['[' + x + ']']} ocorrências)" for x in EXPRESSOES_PROTEGIDAS))

salvar_tabela(por_categoria, "05_exclusoes_por_categoria")
salvar_tabela(por_expressao, "05_exclusoes_por_expressao", index=False)
salvar_tabela(efeito, "05_exclusoes_efeito_por_lema")
salvar_tabela(tokens_excl, "05_tokens_processados_com_exclusoes", index=False)

sem_ocorrencia = [x for c, v in EXCLUSOES.items() for x in v if ocorrencias_excl[x] == 0]
print(f"\nSíntese factual: {por_categoria.loc['Total', 'tokens_retirados']} tokens retirados da contagem de conteúdo "
      f"({por_categoria.loc['Total', 'tokens_de_conteudo_retirados']} eram tokens de conteúdo), em "
      f"{por_categoria.loc['Total', 'ocorrencias']} ocorrências de {len(por_expressao)} expressões; "
      f"{len(efeito)} lemas afetados. Tokens de conteúdo: {num(int(tokens['conteudo'].sum()), 0)} sem exclusões; "
      f"{num(int(tokens_excl['conteudo'].sum()), 0)} com exclusões. O total de tokens (denominador) permanece "
      f"{num(len(tokens_excl), 0)}. Expressões da lista sem ocorrência: {', '.join(sem_ocorrencia) if sem_ocorrencia else 'nenhuma'}.")

**Método e escolhas.** O comando foi traduzido em cinco categorias de expressões, todas listadas em `EXCLUSOES`, na primeira célula desta seção, que pode ser editada:

1. **Construção *led by*:** retira-se a forma *led* quando seguida de *by*. As demais ocorrências do lema *lead* (*lead*, *leading*, *leads*, no sentido de liderar) permanecem, como pede o comando.
2. **Nome de órgão ou instituição governamental:** todas as palavras do nome, por extenso e em sigla (*Department of Commerce* e *DOC*; *National Science Foundation* e *NSF*), inclusive as palavras temáticas que fazem parte do nome (*Science* em *National Science Foundation*; *Security* em *National Security Council*), porque nesses casos elas nomeiam o órgão, e não o tema. Entram também os órgãos que o plano propõe criar (*AI Workforce Research Hub*, *AI Information Sharing and Analysis Center*), a designação *Biden Administration* e as *Armed Forces*. Por ajuste do pesquisador, as menções ao presidente Trump permanecem: *Trump Administration* está em `EXPRESSOES_PROTEGIDAS`, que casa com a busca com prioridade, mas não retira nenhuma palavra; sem essa proteção, *administration* sairia de *Trump Administration* pela designação genérica (item 3). Programas, leis, documentos e cargos não são órgãos e permanecem: *NAIRR*, *CHIPS R&D programs*, *Executive Order*, *Defense Production Act*, *NIST AI Risk Management Framework* (do qual sai apenas a sigla *NIST*), *Chief AI Officers*, *President Trump*.
3. **Designação genérica de órgão ou instituição governamental:** os substantivos que, no texto, designam órgãos ou instituições de governo: *agency/agencies*, *government(s)*, *administration*, *courts*, *law enforcement*, *regulators*, *establishments* e *national laboratories*. Os qualificadores temáticos permanecem (*regulatory* em *regulatory agencies*; *national security* em *national security agencies*).
4. **Federal ligado a órgão ou instituição:** *Federal* é retirado quando qualifica um órgão ou instituição (*Federal agencies*, *Federal government*, *Federal executive councils*, *Federal partners*, *Federal enterprise*, além dos nomes próprios, como *Federal Trade Commission*) e mantido nos demais usos (*Federal funding*, *Federal lands*, *Federal data*, *Federal level*, *Federal Rules of Evidence*). Em *Federal science agencies*, saem *Federal* e *agencies*, e *science* permanece.
5. **Nome do país:** *U.S.*, *United States*, *America* (inclusive em *America’s*) e *American(s)*, em qualquer contexto, inclusive em títulos e em nomes de leis. As formas *U.S.-approved*, *U.S.-origin* e *U.S* (sem o ponto final) já não eram tokens desde a seção 10, porque não seguem o padrão de palavra.

A busca é feita na sequência de palavras de cada frase, sem distinção de maiúsculas e ignorando a pontuação. Em cada posição vale a expressão mais longa, e cada palavra é retirada uma única vez, na categoria da expressão que a capturou; por isso, o nome do país dentro de um nome de órgão (*U.S. Department of Agriculture*) é contado na categoria do país, e o restante do nome, na categoria dos órgãos. A identificação dos órgãos parte do próprio texto (siglas definidas entre parênteses, órgãos indicados como responsáveis pelas medidas) e, quando o texto não basta, do conhecimento geral da estrutura do governo norte-americano, necessário para cumprir o comando.

**Como ler.** A primeira tabela resume o efeito de cada categoria: ocorrências encontradas, palavras retiradas e, entre elas, as que eram tokens de conteúdo (as demais já eram *stopwords*, como *of* e *the*). A segunda lista cada expressão com o seu número de ocorrências; uma expressão com zero ocorrência indicaria erro de grafia na lista. A terceira mostra, para cada lema afetado, a frequência sem e com as exclusões e a categoria responsável. A quarta permite conferir, com o contexto, as ocorrências que **permaneceram** de lemas parcialmente retirados ou protegidos (*federal*, *lead*, *office*, *united*, *states*, *trump*, *administration*). A linha das expressões protegidas informa quantas ocorrências de *Trump Administration* foram mantidas. As linhas finais listam as siglas do texto retiradas e as que ficaram fora da lista.

**Interpretação.** *Observação.* As exclusões retiraram 791 palavras da contagem de conteúdo, em 510 ocorrências das 112 expressões da lista; 699 delas eram tokens de conteúdo, e as demais já eram *stopwords* (como *by* em *led by* e *of* em *Department of Commerce*). Os tokens de conteúdo passam de 5.067 para 4.368, e o total de tokens permanece 8.198. A maior parte vem dos nomes de órgãos e instituições (408 tokens de conteúdo, em 271 ocorrências), seguida do nome do país (135, em 114 ocorrências), de *Federal* ligado a órgão ou instituição (57), das designações genéricas (53) e da construção *led by* (46). Seis lemas do recorte da seção 12 foram zerados: *doc* (49 ocorrências), *agency* (39), *american* (35), *government* (30), *america* (26) e *u.s.* (25). Outros dois foram reduzidos: *lead*, de 59 para 13, e *federal*, de 49 para 18. Palavras temáticas também perderam ocorrências por fazerem parte de nomes de órgãos: *national* (de 40 para 29), *security* (de 34 para 30), *technology* (de 37 para 34), *science* (de 14 para 11), *intelligence* (de 11 para 7) e *ai* (de 244 para 240). Por ajuste do pesquisador, *Trump Administration* (7 ocorrências) está protegida: as 13 ocorrências de *trump* permanecem, e *administration* passa de 17 para 7, que são justamente as de *Trump Administration*. Todas as expressões da lista têm ao menos uma ocorrência, e as ocorrências mantidas de *federal* (18) e de *lead* (13) correspondem, no contexto, a usos não ligados a instituições e ao sentido de liderar. *Inferência.* Cerca de um sétimo dos tokens de conteúdo (699 de 5.067) refere-se a órgãos, ao governo, ao próprio país ou à fórmula de atribuição de responsabilidade. Parte relevante do vocabulário mais frequente da base sem exclusões (seção 12) era, portanto, vocabulário de atribuição institucional e de autorreferência, e não de tema.

**Fragilidades e ajustes possíveis.** A fronteira entre órgão e não órgão exige julgamento em alguns casos, que ficaram assim decididos e podem ser revertidos na lista: (a) organismos internacionais (*United Nations*, *Organisation for Economic Co-operation and Development*, *International Telecommunication Union*) permanecem, por não serem órgãos de governo de um Estado; (b) estados norte-americanos (*state*) permanecem, por serem entes federativos, e não órgãos; (c) pessoas e cargos permanecem (*President Trump*, *Vice President Vance*, *Chief AI Officers*), e, por ajuste do pesquisador, também *Trump Administration*; já *Biden Administration* e as referências sem nome ao governo (*the Administration*, *this Administration*, *the previous administration*) continuam retiradas, por não mencionarem o presidente Trump; (d) instituições de ensino e pesquisa e estruturas cuja natureza o texto não define permanecem (*Senior Military Colleges*, *Federally Funded Research and Development Centers*, *Cooperative Ecosystem Studies Units*, *AI Centers of Excellence*, *National Secure Data Service*, *NIST AI Consortium*, exceto a sigla *NIST*); (e) *Nation*, em *our Nation’s workers*, permanece, porque se refere ao país sem nomeá-lo; (f) os órgãos que o plano propõe criar saem, embora sejam também conteúdo programático do plano. A lista é fechada: um órgão mencionado no texto e ausente da lista continua na contagem; a tabela de resíduos e a lista de siglas servem para essa conferência.

## 16. Lemas de conteúdo mais frequentes, com as exclusões

*Visualização construída por comando do pesquisador (2026-10-05) e ajustada por novo comando no mesmo dia (menções ao presidente Trump mantidas): exclusões da seção 15 e recorte de 30 lemas.*

In [ ]:
freq_excl = tabela_frequencias(tokens_excl)
salvar_tabela(freq_excl, "06_frequencias_lemas_com_exclusoes_completa")
relatar_empates(freq_excl, "frequencia", TOP_N_LEMAS_EXCLUSOES, "lema")
top_excl = freq_excl.head(TOP_N_LEMAS_EXCLUSOES)

fig, ax = plt.subplots(figsize=(7.5, 0.27 * len(top_excl) + 1.3))
y = np.arange(len(top_excl))
ax.barh(y, top_excl["frequencia"], height=0.7, color=COR)
ax.set_yticks(y, labels=top_excl.index)
ax.invert_yaxis()
xmax = top_excl["frequencia"].max()
for yi, (_, linha) in zip(y, top_excl.iterrows()):
    ax.text(linha["frequencia"] + xmax * 0.015, yi,
            f"{num(linha['por_1000_tokens'])} por 1.000 · {int(linha['unidades_com_ocorrencia'])}/{N_UNIDADES} unid.",
            va="center", fontsize=9, color=COR_TINTA_2)
ax.set_xlim(0, xmax * 1.55)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.xaxis.set_major_formatter(FMT_BR)
ax.set_xlabel("Frequência absoluta (ocorrências)")
ax.set_title(f"{SIGLA} · {len(top_excl)} lemas de conteúdo mais frequentes, com exclusões", loc="left")
ax.grid(axis="y", visible=False)
salvar_figura(fig, "06_lemas_mais_frequentes_com_exclusoes")
plt.show()

# Comparação com a base sem exclusões (seção 12): frequência e posição no ranking
posicao_sem = pd.Series(np.arange(1, len(freq) + 1), index=freq.index)
tabela_top_excl = top_excl[["frequencia", "por_1000_tokens", "unidades_com_ocorrencia", "pct_unidades",
                            "DP_norm", "formas_observadas"]].assign(
    frequencia_sem_exclusoes=freq["frequencia"].reindex(top_excl.index),
    posicao_sem_exclusoes=posicao_sem.reindex(top_excl.index))
exibir(tabela_top_excl)
n_conteudo_excl = int(tokens_excl["conteudo"].sum())
novos = [l for l in top_excl.index if posicao_sem[l] > TOP_N_LEMAS]
saidos = [l for l in freq.head(TOP_N_LEMAS).index if l not in top_excl.index]
print(f"Síntese factual: os {len(top_excl)} lemas acima somam {num(top_excl['frequencia'].sum(), 0)} ocorrências, "
      f"{num(top_excl['frequencia'].sum() / n_conteudo_excl * 100, 1)}% dos {num(n_conteudo_excl, 0)} tokens de conteúdo "
      f"com exclusões; o documento tem {num(len(freq_excl), 0)} lemas de conteúdo distintos com exclusões. "
      f"Lemas do recorte que estavam fora dos {TOP_N_LEMAS} primeiros na seção 12: {', '.join(novos) if novos else 'nenhum'}. "
      f"Lemas dos {TOP_N_LEMAS} primeiros da seção 12 que saíram do recorte: {', '.join(saidos) if saidos else 'nenhum'}.")

# Termos consultados pelo pesquisador: frequência e posição no ranking completo com exclusões
posicao_excl = pd.Series(np.arange(1, len(freq_excl) + 1), index=freq_excl.index)
print("Termos consultados pelo pesquisador: " + "; ".join(
    f"{t}: {int(freq_excl.loc[t, 'frequencia'])} ocorrências, posição {posicao_excl[t]} de {num(len(freq_excl), 0)} "
    f"(formas: {freq_excl.loc[t, 'formas_observadas']})" if t in freq_excl.index
    else f"{t}: sem ocorrência como token de conteúdo" for t in TERMOS_CONSULTADOS)
    + f". O {TOP_N_LEMAS_EXCLUSOES}º lema do recorte tem {num(top_excl['frequencia'].iloc[-1], 0)} ocorrências.")

**Método e escolhas.** Mesma medida da seção 12, calculada sobre os tokens de conteúdo que restam após as exclusões da seção 15. O recorte passa de 25 para 30 lemas (`TOP_N_LEMAS_EXCLUSOES`), por comando do pesquisador. O denominador das taxas por 1.000 tokens continua sendo o total de tokens do documento, de modo que as taxas são comparáveis às da seção 12; o alcance (unidades com ocorrência) e o DP são recalculados com as exclusões. A tabela acrescenta duas colunas de comparação: a frequência do lema sem as exclusões e a sua posição no ranking sem exclusões (seção 12). Empates no ponto de corte são resolvidos por ordem alfabética e informados acima do gráfico.

**Como ler.** Como na seção 12. Quando a frequência com exclusões é menor que a frequência sem exclusões, parte das ocorrências do lema estava em nomes de órgãos, em designações de governo ou no nome do país (ver a tabela de lemas afetados, na seção 15).

**Interpretação.** *Observação.* Com as exclusões, *ai* segue isolado no topo: 240 ocorrências (29,28 por 1.000 tokens), em 134 das 184 unidades, com DP de 0,31. Abaixo dele vêm *system* (52), *new* (42), *use* (39), *data* (36), *technology* (34), *infrastructure* e *model* (33 cada), *development*, *ensure* e *include* (32 cada), *security* (30) e *national* (29). Os 30 lemas somam 1.002 ocorrências, 22,9% dos 4.368 tokens de conteúdo com exclusões. Em relação aos 25 primeiros da seção 12, saíram *lead*, *doc*, *agency*, *american*, *government*, *america* e *u.s.*. Com a saída desses lemas e o recorte ampliado para 30, entraram *adoption* (22), *innovation* (20), *create*, *power* e *workforce* (19 cada), *grid*, *industry* e *need* (18 cada), *critical* e *tool* (17 cada), *action* e *exist* (16 cada). *federal* permanece no recorte com 18 ocorrências (eram 49), todas em usos não ligados a instituições. Os lemas mais concentrados do recorte são *grid* (18 ocorrências em 7 unidades; DP 0,94) e *control* (26 em 16; DP 0,92). Quatro lemas com 16 ocorrências ficaram fora do corte por ordem alfabética: *manufacturing*, *risk*, *semiconductor* e *standard*. Os termos consultados pelo pesquisador estão abaixo do corte de 16 ocorrências: *trump* tem 13 (posição 56 do ranking completo com exclusões), todas mantidas por ajuste do pesquisador, e *race* tem 9 (posição 99), das quais 8 na forma *race* e 1 em *raced*. *race* não está na lista de *stopwords*; fica fora do recorte apenas pela frequência. *Inferência.* Sem a camada de atribuição institucional e de autorreferência nacional, o vocabulário frequente aproxima-se dos objetos de política do plano: a tecnologia e os seus componentes (*system*, *model*, *data*, *technology*), a base material (*infrastructure*, *power*, *grid*, *build*), a pesquisa, a inovação e a adoção (*research*, *development*, *innovation*, *adoption*), a força de trabalho (*workforce*) e a segurança (*security*, *national*, *control*, *critical*). Verbos como *use*, *ensure*, *include*, *need* e *create* são de uso geral na redação de medidas e indicam mais a forma prescritiva do texto do que um tema.

**Fragilidades e ajustes possíveis.** As exclusões mudam o que o recorte mostra, e não o texto: os termos retirados continuam no plano e no denominador. O ranking com exclusões não deve ser lido isoladamente; a comparação com a seção 12 mostra quanto do vocabulário mais frequente do plano se refere a órgãos, ao governo e ao próprio país. Os verbos de redação que agora sobem no recorte (*use*, *ensure*, *include*, *need*, *create*) podem ser retirados, se o pesquisador quiser um recorte só de termos temáticos; essa decisão não foi tomada aqui, porque não consta do comando. Valem também as fragilidades da seção 12.

## 17. Distribuição dos lemas ao longo do documento, com as exclusões

*Visualização construída por comando do pesquisador (2026-10-05) e ajustada por novo comando no mesmo dia (menções ao presidente Trump mantidas): exclusões da seção 15; recorte de 15 lemas mantido.*

In [ ]:
lemas_disp_excl = freq_excl.head(TOP_N_DISPERSAO).index.tolist()
# Mesmas linhas de referência da seção 13 (início de cada título interno): as exclusões
# não alteram as posições, porque o total de tokens não muda.
inicio_unidade_excl = tokens_excl.groupby("unidade")["posicao"].min() / N_TOKENS
inicios_excl = (inicio_unidade_excl.reindex(ids_titulos) if ids_titulos else inicio_unidade_excl).dropna().to_numpy()
posicoes_excl = [tokens_excl.loc[tokens_excl["conteudo"] & (tokens_excl["lema"] == lema), "posicao"].to_numpy() / N_TOKENS
                 for lema in lemas_disp_excl]
rotulos_excl = [f"{lema}  (DP = {num(freq_excl.loc[lema, 'DP_norm'])})" for lema in lemas_disp_excl]

fig, ax = plt.subplots(figsize=(7.5, 0.33 * len(lemas_disp_excl) + 1.3))
grafico_dispersao_posicional(ax, posicoes_excl, rotulos_excl, inicios_excl, COR)
ax.set_title(f"{SIGLA} · posição das ocorrências dos {len(lemas_disp_excl)} lemas mais frequentes, com exclusões", loc="left")
salvar_figura(fig, "07_dispersao_lemas_com_exclusoes")
plt.show()

por_quinto_excl = pd.DataFrame([np.histogram(p, bins=np.linspace(0, 1, 6))[0] for p in posicoes_excl],
                               index=pd.Index(lemas_disp_excl, name="lema"), columns=QUINTOS)
disp_excl = freq_excl.head(TOP_N_DISPERSAO)[["frequencia", "unidades_com_ocorrencia", "pct_unidades", "DP_norm"]].join(por_quinto_excl)
exibir(disp_excl.sort_values("DP_norm"), titulo="Dispersão e ocorrências por quinto do texto (posição relativa), com exclusões")
salvar_tabela(disp_excl, "07_dispersao_lemas_com_exclusoes")
novos_disp = [l for l in lemas_disp_excl if l not in lemas_disp]
print(f"Síntese factual: DP normalizado entre {num(disp_excl['DP_norm'].min())} e {num(disp_excl['DP_norm'].max())} "
      f"nos {len(lemas_disp_excl)} lemas; linhas de referência no início de {len(inicios_excl)} títulos internos "
      f"(as mesmas da seção 13). Lemas que não estavam entre os {TOP_N_DISPERSAO} da seção 13: "
      f"{', '.join(novos_disp) if novos_disp else 'nenhum'}.")

**Método e escolhas.** Mesmo gráfico e mesmas medidas da seção 13 (posição relativa, DP normalizado e ocorrências por quinto do texto), para os 15 lemas mais frequentes após as exclusões (`TOP_N_DISPERSAO`, mantido). As posições não mudam com as exclusões, porque o total de tokens é o mesmo; muda apenas o conjunto de ocorrências contadas. As linhas de referência são as mesmas da seção 13 (início de cada título interno), identificadas na tabela daquela seção.

**Como ler.** Como na seção 13: traços espalhados indicam termos presentes em todo o texto; traços agrupados, termos concentrados em uma parte. A tabela ordena os lemas do mais espalhado ao mais concentrado.

**Interpretação.** *Observação.* Entre os 15 lemas mais frequentes com exclusões, *ai* continua sendo o único espalhado por todo o texto (DP 0,31; entre 35 e 61 ocorrências por quinto). Em relação à seção 13, entraram no recorte *model*, *ensure*, *include*, *program* e *control*. *control* é o mais concentrado (DP 0,92): tem 20 das 26 ocorrências no último quinto (0,8–1) e nenhuma nos quintos 0–0,2 e 0,6–0,8. No gráfico, esse grupo fica entre os títulos *Strengthen AI Compute Export Control Enforcement* (0,880) e *Ensure that the U.S. Government is at the Forefront of Evaluating National Security Risks in Frontier Models* (0,948). *model* tem 15 das 33 ocorrências no primeiro quinto, agrupadas entre os títulos *Encourage Open-Source and Open-Weight AI* (0,148) e *Enable AI Adoption* (0,194). *program* tem 12 das 27 ocorrências entre 0,6 e 0,8, faixa que inclui o título *Train a Skilled Workforce for AI Infrastructure* (0,705). *ensure* se divide entre o início e o fim do texto (10, 1, 2, 10 e 9 ocorrências por quinto). *infrastructure* mantém a concentração vista na seção 13 (18 das 33 ocorrências entre 0,6 e 0,8), e *security* tem 13 das 30 no último quinto. *Inferência.* A retirada dos nomes de órgãos e do nome do país não altera o padrão observado na seção 13: poucos termos atravessam o texto, e a maior parte dos lemas frequentes concentra-se nas seções em que o seu tema é tratado. Os lemas que entraram no recorte reforçam esse padrão: *control* pertence quase só às seções finais, sobre controles de exportação, e *model*, à seção sobre modelos abertos.

**Fragilidades e ajustes possíveis.** Valem as fragilidades da seção 13: o DP é sensível à segmentação e à frequência, e os quintos são faixas de posição, e não divisões do plano.

## 18. Expressões recorrentes (bigramas), com as exclusões

*Visualização construída por comando do pesquisador (2026-10-05) e ajustada por novo comando no mesmo dia (menções ao presidente Trump mantidas): exclusões da seção 15; recorte de 20 bigramas mantido.*

In [ ]:
bigramas_excl = tabela_bigramas(tokens_excl)
salvar_tabela(bigramas_excl, "08_bigramas_com_exclusoes_completa")
elegiveis_excl = bigramas_excl[bigramas_excl["frequencia"] >= FREQ_MINIMA_BIGRAMA]
relatar_empates(elegiveis_excl, "frequencia", TOP_N_BIGRAMAS, "bigrama")
topb_excl = elegiveis_excl.head(TOP_N_BIGRAMAS)

if topb_excl.empty:
    print(f"Nenhum bigrama atinge a frequência mínima de {FREQ_MINIMA_BIGRAMA}; gráfico não gerado.")
else:
    fig, ax = plt.subplots(figsize=(7.5, 0.27 * len(topb_excl) + 1.3))
    y = np.arange(len(topb_excl))
    ax.barh(y, topb_excl["frequencia"], height=0.7, color=COR)
    ax.set_yticks(y, labels=topb_excl["forma_principal"])
    ax.invert_yaxis()
    xmax = topb_excl["frequencia"].max()
    for yi, (_, linha) in zip(y, topb_excl.iterrows()):
        ax.text(linha["frequencia"] + xmax * 0.015, yi,
                f"{int(linha['unidades_com_ocorrencia'])}/{N_UNIDADES} unid.",
                va="center", fontsize=9, color=COR_TINTA_2)
    ax.set_xlim(0, xmax * 1.25)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.xaxis.set_major_formatter(FMT_BR)
    ax.set_xlabel("Frequência absoluta (ocorrências)")
    ax.set_title(f"{SIGLA} · {len(topb_excl)} expressões de duas palavras mais frequentes, com exclusões", loc="left")
    ax.grid(axis="y", visible=False)
    salvar_figura(fig, "08_bigramas_com_exclusoes")
    plt.show()
    exibir(topb_excl.assign(frequencia_sem_exclusoes=bigramas["frequencia"].reindex(topb_excl.index)))
saidos_b = [b for b in elegiveis.head(TOP_N_BIGRAMAS).index if b not in topb_excl.index]
print(f"Síntese factual: {num(len(bigramas_excl), 0)} bigramas distintos; {num(len(elegiveis_excl), 0)} com frequência "
      f">= {FREQ_MINIMA_BIGRAMA}. Bigramas dos {TOP_N_BIGRAMAS} da seção 14 que saíram do recorte: "
      f"{', '.join(saidos_b) if saidos_b else 'nenhum'}.")

**Método e escolhas.** Mesma medida da seção 14, calculada com as exclusões da seção 15: um par só é contado se as duas palavras continuarem como tokens de conteúdo. Como a adjacência é verificada no texto original, a retirada de uma palavra não une as palavras vizinhas a ela. O recorte de 20 bigramas (`TOP_N_BIGRAMAS`) é mantido, por comando do pesquisador. A tabela acrescenta a frequência de cada bigrama sem as exclusões (seção 14).

**Como ler.** Como na seção 14. Os bigramas que saíram do recorte, em relação à seção 14, são informados na síntese factual.

**Interpretação.** *Observação.* Com as exclusões, há 1.514 bigramas distintos (eram 1.712), dos quais 170 têm frequência igual ou superior a 2 (eram 196). Saíram do recorte os seis bigramas formados por nomes de instituições ou pelo nome do país: *united states*, *federal agency*, *federal government*, *u.s. government*, *american ai* e *american worker*. Os 14 que permaneceram mantêm a frequência da seção 14, com uma exceção: *national security* passa de 17 para 16, porque uma ocorrência estava no nome *National Security Council*. Entre eles está *trump administration* (7 ocorrências, em 5 unidades), mantido por ajuste do pesquisador. Entraram, todos com 5 ocorrências, *ai compute*, *ai development*, *ai skill*, *ai technologies*, *frontier ai* e *president trump*; *security risk* e *skill development*, também com 5, ficaram fora por ordem alfabética. *ai systems* (32 ocorrências, em 21 unidades) tem o dobro do segundo bigrama, *national security* (16). *Inferência.* Sem os nomes de órgãos e do país, as expressões mais repetidas são quase todas temáticas: 10 dos 20 bigramas contêm *ai*, e os demais tratam de segurança (*national security*, *critical infrastructure*), de infraestrutura e indústria (*data centers*, *semiconductor manufacturing*, *export controls*, *private sector*) e da própria natureza do documento (*action plan*, *executive order*). As duas expressões restantes, *trump administration* e *president trump*, referem-se ao presidente e ao seu governo e foram mantidas por ajuste do pesquisador.

**Fragilidades e ajustes possíveis.** Valem as fragilidades da seção 14. A retirada dos nomes de órgãos retira também os bigramas formados dentro desses nomes (por exemplo, *national security* em *National Security Council*).